<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_2/Word2Vec_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Word2Vec: нейросетевые эмбеддинги слов

## Введение: почему нейросетевые подходы изменили NLP

### От частотных методов к нейросетевым: что изменилось

К моменту появления Word2Vec в 2013 году в арсенале исследователей уже был целый набор методов векторизации текста. One-hot encoding, Bag of Words, TF-IDF, LSA, pLSA, LDA — все они решали задачу перевода текста в числовую форму. Но у них был общий фундаментальный недостаток: они не давали **плотных семантически насыщенных векторов отдельных слов**, которые можно было бы эффективно использовать в нейросетевых архитектурах и downstream-задачах.

Давайте разберём этот недостаток подробно, потому что понимание того, чего не хватало предыдущим методам, критически важно для понимания того, что именно предложил Word2Vec.

One-hot encoding даёт каждому слову вектор размерности $N$, где $N$ — размер словаря. В этом векторе ровно одна компонента равна единице, остальные — нулю. Все слова ортогональны друг другу: скалярное произведение любых двух разных one-hot векторов равно нулю. Это означает, что с точки зрения математики все слова одинаково далеки друг от друга. Слово «кошка» не ближе к слову «собака», чем к слову «диван» или «на». Семантическая информация, присущая языку, полностью теряется.

Bag of Words и TF-IDF дают векторы документов, а не отдельных слов. Они позволяют сравнивать документы по их тематической близости, но не дают представления отдельных слов. Более того, эти векторы разрежены и высокоразмерны: если словарь содержит миллион слов, каждый вектор документа имеет миллион компонент, из которых ненулевыми являются лишь несколько десятков или сотен.

LSA и LDA дают плотные векторы, но с оговорками. LSA — это алгебраический метод, основанный на сингулярном разложении матрицы «термин-документ». Он не имеет вероятностной интерпретации, и его латентные факторы трудно интерпретировать содержательно. LDA — это вероятностная тематическая модель, где векторы слов — это распределения по темам. Но темы — это не то же самое, что семантические компоненты значения слова. Кроме того, LDA требует задания числа тем $K$ и вычислительно сложна.

Все эти методы объединяет ещё одна общая проблема: они не используют **контекст**. Значение слова в них определяется либо его индексом (one-hot), либо частотой (BoW, TF-IDF), либо распределением по темам (LDA). Но в естественном языке значение слова определяется его **окружением** — словами, которые встречаются рядом с ним. Слово «банк» в контексте «река» означает одно, а в контексте «деньги» — совсем другое. Ни один из перечисленных методов не улавливает эту контекстную зависимость.

### Дистрибутивная гипотеза как фундамент

В основе Word2Vec лежит **дистрибутивная гипотеза**, сформулированная лингвистами Джоном Фёрсом и Зеллигом Харрисом в 1950-х годах. Она утверждает:

> Слова, встречающиеся в похожих контекстах, имеют похожие значения.

Эта гипотеза имеет глубокий интуитивный смысл. Рассмотрим слова «кошка» и «собака». Оба они могут встречаться в предложениях «___ сидит на окне», «___ спит на диване», «___ ест из миски». Их контексты пересекаются, потому что оба слова обозначают домашних животных. Слово «диван», напротив, встречается в контекстах «сидеть на ___», «лежать на ___», «мягкий ___», которые не пересекаются с контекстами «кошки» и «собаки». А слово «сидит» встречается в контекстах «кошка ___», «собака ___», «человек ___», которые частично пересекаются с контекстами животных, но не полностью.

Формализуем эту идею. Пусть $w$ — целевое слово, $c$ — контекстное слово (слово в окне вокруг $w$). Дистрибутивная гипотеза утверждает, что распределение $P(c \mid w)$ несёт информацию о значении $w$. Если два слова $w_i$ и $w_j$ имеют похожие распределения контекстов $P(c \mid w_i) \approx P(c \mid w_j)$, то они семантически близки.

Word2Vec формализует эту идею через задачу предсказания контекста. Мы обучаем нейросеть предсказывать, какие слова встречаются рядом с данным словом. В процессе обучения сеть вынуждена сжимать информацию о значении слова в вектор небольшой размерности. Именно это сжатие порождает семантическую структуру.

### Ключевая идея Word2Vec: обучение через предсказание

Представьте, что вы пытаетесь описать значение слова, используя только 300 чисел. Вы не можете запомнить все контексты, в которых слово встречается. Вместо этого вы вынуждены выделить **общие черты** этих контекстов. Например, для слова «кошка» вы запомните, что оно связано с животными, домашними питомцами, мягкостью, независимостью. Эти общие черты и есть семантические компоненты, которые кодируются в векторе.

Word2Vec реализует эту идею через простую нейросетевую архитектуру. Мы берём корпус текста и для каждого слова пытаемся предсказать его контекст (или по контексту предсказать слово). Если модель хорошо справляется с этой задачей, значит, вектор слова содержит достаточно информации о его значении.

Важно понимать, что Word2Vec не пытается явно смоделировать значение слова. Он не строит онтологию, не использует словари, не размечает данные. Он просто предсказывает контекст. И семантическая структура возникает **автоматически**, как побочный продукт решения этой задачи.

### Две архитектуры Word2Vec

Word2Vec имеет две архитектуры, которые решают одну и ту же задачу, но с разных сторон:

1. **CBOW (Continuous Bag of Words):** по контекстным словам предсказывает целевое слово.
2. **Skip-gram:** по целевому слову предсказывает контекстные слова.

Рассмотрим пример. Пусть корпус содержит предложение «кошка сидит на окне», и мы используем окно $m = 1$.

- **CBOW:** по словам «кошка» и «на» предсказывает слово «сидит».
- **Skip-gram:** по слову «сидит» предсказывает слова «кошка» и «на».

Обе архитектуры используют одну и ту же идею: обучать векторы через предсказание. Разница в направлении предсказания. CBOW усредняет контекст и предсказывает одно слово, что делает её быстрой. Skip-gram использует одно слово для предсказания нескольких контекстных слов, что делает её более чувствительной к редким словам.

Мы начнём с подробного разбора CBOW, а затем перейдём к Skip-gram.

---

# Архитектура CBOW (Continuous Bag of Words)

## 1. Формальное определение задачи

### 1.1 Постановка задачи и обозначения

Пусть дан корпус — последовательность слов:

$$
w_1, w_2, \ldots, w_T,
$$

где $T$ — длина корпуса (общее число слов). Словарь:

$$
V = \{w_1, w_2, \ldots, w_N\},
$$

где $N = |V|$ — размер словаря. Зафиксируем размер окна $m$. Для каждой позиции $t$ целевое слово — это $w_t$, а контекстные слова — это слова в окне вокруг $w_t$:

$$
w_{t-m}, \ldots, w_{t-1}, w_{t+1}, \ldots, w_{t+m}.
$$

Обозначим контекст через $C_t$:

$$
C_t = \{w_{t-m}, \ldots, w_{t-1}, w_{t+1}, \ldots, w_{t+m}\}.
$$

Число контекстных слов равно $2m$.

Рассмотрим конкретный пример. Пусть корпус — это объединённая последовательность наших трёх документов:

$$
\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{сидит}, \text{на}, \text{крыльце}, \text{кошка}, \text{спит}, \text{на}, \text{диване}.
$$

Длина последовательности $T = 12$. Пусть окно $m = 1$. Тогда для каждой позиции $t$ контекст — это слова на позициях $t-1$ и $t+1$.

| Позиция $t$ | Целевое $w_t$ | Контекст $C_t$ |
|-------------|---------------|----------------|
| 1 | кошка | $\{$сидит$\}$ |
| 2 | сидит | $\{$кошка, на$\}$ |
| 3 | на | $\{$сидит, окне$\}$ |
| 4 | окне | $\{$на, собака$\}$ |
| 5 | собака | $\{$окне, сидит$\}$ |
| 6 | сидит | $\{$собака, на$\}$ |
| 7 | на | $\{$сидит, крыльце$\}$ |
| 8 | крыльце | $\{$на, кошка$\}$ |
| 9 | кошка | $\{$крыльце, спит$\}$ |
| 10 | спит | $\{$кошка, на$\}$ |
| 11 | на | $\{$спит, диване$\}$ |
| 12 | диване | $\{$на$\}$ |

**Задача CBOW:** для каждой позиции $t$ предсказать целевое слово $w_t$ по контексту $C_t$. Мы хотим, чтобы модель присваивала высокую вероятность правильному целевому слову.

### 1.2 Параметры модели

CBOW имеет два набора параметров:

1. **Входные векторы (input vectors):** матрица $U \in \mathbb{R}^{N \times d}$, где $d$ — размерность эмбеддинга. Строка $i$ матрицы $U$ — это вектор $u_{w_i} \in \mathbb{R}^d$ слова $w_i$. Эти векторы используются для контекстных слов.

2. **Выходные векторы (output vectors):** матрица $V \in \mathbb{R}^{N \times d}$. Строка $i$ матрицы $V$ — это вектор $v_{w_i} \in \mathbb{R}^d$ слова $w_i$. Эти векторы используются для целевого слова.

Почему нужны два набора векторов? Это тонкий момент, который часто вызывает вопросы. Дело в том, что роли слова в паре (контекстное/целевое) асимметричны. Если бы мы использовали один вектор $w_w$ для каждого слова, то вероятность $P(w_t \mid C_t)$ была бы симметричной относительно перестановки целевого и контекстного слов. Но в реальности это не так: вероятность встретить «кошку» рядом с «сидит» не равна вероятности встретить «сидит» рядом с «кошкой». Слово «сидит» встречается вообще чаще, чем «кошка», поэтому его вероятность как контекстного слова выше. Разделение на входные и выходные векторы позволяет модели уловить эту асимметрию.

После обучения обычно используют либо $u_w$, либо $v_w$, либо их сумму $u_w + v_w$ как итоговый эмбеддинг. На практике $u_w$ и $v_w$ дают близкие результаты, но $u_w$ используется чаще.

### 1.3 Прямой проход: шаг за шагом

Рассмотрим, как CBOW вычисляет вероятность целевого слова по контексту. Мы разберём каждый шаг подробно, с формулами и примером.

#### Шаг 1: Получение векторов контекстных слов

Для каждого контекстного слова $c \in C_t$ берём его входной вектор $u_c \in \mathbb{R}^d$. Это просто выборка строки из матрицы $U$ по индексу слова.

**Пример:** контекст $C_t = \{$кошка, на$\}$. Пусть

$$
u_{\text{кошка}} = (0.2, -0.1), \quad u_{\text{на}} = (-0.1, 0.6).
$$

Здесь $d = 2$ для наглядности. В реальных задачах $d = 100$–$300$.

#### Шаг 2: Усреднение (или суммирование)

Вычисляем **усреднённый вектор контекста**:

$$
h = \frac{1}{2m} \sum_{c \in C_t} u_c \in \mathbb{R}^d.
$$

Разберём эту формулу по частям. Сумма $\sum_{c \in C_t} u_c$ — это покомпонентная сумма векторов всех контекстных слов:

$$
\sum_{c \in C_t} u_c = u_{c_1} + u_{c_2} + \ldots + u_{c_{2m}}.
$$

Каждый вектор $u_{c_i}$ имеет размерность $d$. Сумма — тоже вектор размерности $d$, где каждая компонента — это сумма соответствующих компонент:

$$
\left( \sum_{c \in C_t} u_c \right)_k = \sum_{c \in C_t} (u_c)_k, \quad k = 1, \ldots, d.
$$

Деление на $2m$ — это нормировка. Она нужна, чтобы вектор $h$ не зависел от размера окна. Если мы используем суммирование без нормировки, то при большом окне вектор $h$ будет иметь большую норму, что повлияет на масштаб скалярных произведений и, следовательно, на градиенты. Усреднение делает вектор $h$ сопоставимым по норме с отдельными векторами слов.

**Пример:**

$$
h = \frac{1}{2} \left[ (0.2, -0.1) + (-0.1, 0.6) \right] = \frac{1}{2} (0.1, 0.5) = (0.05, 0.25).
$$

Тонкий момент: в оригинальной статье Word2Vec используется **суммирование**, а не усреднение:

$$
h = \sum_{c \in C_t} u_c.
$$

Разница невелика: усреднение просто масштабирует $h$ на константу $1/(2m)$, что можно учесть в скорости обучения. Мы будем использовать усреднение, потому что оно делает вектор $h$ сопоставимым по норме с отдельными векторами слов, что удобнее для анализа и визуализации.

#### Шаг 3: Вычисление оценки для каждого слова словаря

Для каждого слова $w \in V$ вычисляем **скалярное произведение**:

$$
s_w = v_w^\top h = \sum_{k=1}^{d} v_{w,k} \cdot h_k.
$$

Разберём формулу. $v_w \in \mathbb{R}^d$ — выходной вектор слова $w$. $h \in \mathbb{R}^d$ — усреднённый вектор контекста. Скалярное произведение — это сумма покомпонентных произведений:

$$
v_w^\top h = v_{w,1} \cdot h_1 + v_{w,2} \cdot h_2 + \ldots + v_{w,d} \cdot h_d.
$$

Результат — скаляр $s_w$, который является мерой совместимости слова $w$ с контекстом.

**Интуиция:** если выходной вектор слова $w$ близок к вектору контекста $h$ (то есть указывает в том же направлении), скалярное произведение велико и положительно. Это означает, что слово $w$ хорошо «подходит» к данному контексту. Если векторы ортогональны, скалярное произведение равно нулю. Если противоположны, отрицательно.

**Пример:** для слова «сидит» с $v_{\text{сидит}} = (-0.2, 0.4)$:

$$
s_{\text{сидит}} = (-0.2) \cdot 0.05 + 0.4 \cdot 0.25 = -0.01 + 0.10 = 0.09.
$$

#### Шаг 4: Softmax

Вероятность целевого слова $w_t$ при условии контекста $C_t$:

$$
P(w_t \mid C_t) = \frac{\exp(s_{w_t})}{\sum_{w \in V} \exp(s_w)} = \frac{\exp(v_{w_t}^\top h)}{\sum_{w \in V} \exp(v_w^\top h)}.
$$

Разберём формулу по частям.

**Числитель:** $\exp(s_{w_t}) = \exp(v_{w_t}^\top h)$. Экспонента делает значение положительным и усиливает различия. Если $s_{w_t}$ велико, $\exp(s_{w_t})$ очень велико. Если мало, близко к нулю.

**Знаменатель:** $\sum_{w \in V} \exp(s_w) = \sum_{w \in V} \exp(v_w^\top h)$. Это сумма экспонент по всему словарю. Она нормирует вероятность так, чтобы сумма по всем возможным целевым словам была равна единице.

**Свойства softmax:**

- $P(w_t \mid C_t) > 0$ для всех $w_t$;
- $\sum_{w \in V} P(w \mid C_t) = 1$;
- Если $s_{w_t}$ велико по сравнению с другими $s_w$, то $P(w_t \mid C_t) \approx 1$.

**Пример:** пусть словарь $\{$кошка, сидит, на, окне$\}$, $h = (0.05, 0.25)$.

Выходные векторы:

$$
v_{\text{кошка}} = (0.1, 0.3), \quad v_{\text{сидит}} = (-0.2, 0.4), \quad v_{\text{на}} = (0.5, -0.1), \quad v_{\text{окне}} = (0.3, 0.2).
$$

Скалярные произведения:

$$
s_{\text{кошка}} = 0.1 \cdot 0.05 + 0.3 \cdot 0.25 = 0.005 + 0.075 = 0.080,
$$

$$
s_{\text{сидит}} = -0.2 \cdot 0.05 + 0.4 \cdot 0.25 = -0.010 + 0.100 = 0.090,
$$

$$
s_{\text{на}} = 0.5 \cdot 0.05 + (-0.1) \cdot 0.25 = 0.025 - 0.025 = 0.000,
$$

$$
s_{\text{окне}} = 0.3 \cdot 0.05 + 0.2 \cdot 0.25 = 0.015 + 0.050 = 0.065.
$$

Экспоненты:

$$
\exp(0.080) \approx 1.083, \quad \exp(0.090) \approx 1.094, \quad \exp(0.000) = 1.000, \quad \exp(0.065) \approx 1.067.
$$

Сумма:

$$
Z = 1.083 + 1.094 + 1.000 + 1.067 = 4.244.
$$

Вероятности:

$$
P(\text{кошка} \mid C) = 1.083 / 4.244 \approx 0.255,
$$

$$
P(\text{сидит} \mid C) = 1.094 / 4.244 \approx 0.258,
$$

$$
P(\text{на} \mid C) = 1.000 / 4.244 \approx 0.236,
$$

$$
P(\text{окне} \mid C) = 1.067 / 4.244 \approx 0.251.
$$

**Наблюдение:** все вероятности близки к $1/4 = 0.25$, потому что векторы ещё не обучены. После обучения вероятности станут более контрастными: правильное целевое слово будет иметь высокую вероятность, остальные — низкие.

---

## 2. Функция правдоподобия и её вывод

### 2.1 Правдоподобие для одной позиции

Для одной позиции $t$ вероятность целевого слова:

$$
P(w_t \mid C_t) = \frac{\exp(v_{w_t}^\top h_t)}{\sum_{w \in V} \exp(v_w^\top h_t)},
$$

где $h_t = \frac{1}{2m} \sum_{c \in C_t} u_c$.

**Логарифм правдоподобия** для одной позиции:

$$
\log P(w_t \mid C_t) = v_{w_t}^\top h_t - \log \sum_{w \in V} \exp(v_w^\top h_t).
$$

Разберём эту формулу. Мы взяли логарифм от дроби:

$$
\log \frac{\exp(v_{w_t}^\top h_t)}{\sum_{w} \exp(v_w^\top h_t)} = \log \exp(v_{w_t}^\top h_t) - \log \sum_{w} \exp(v_w^\top h_t).
$$

Поскольку $\log \exp(x) = x$, первое слагаемое равно $v_{w_t}^\top h_t$. Второе слагаемое — логарифм суммы экспонент.

**Интерпретация:**

- Первое слагаемое $v_{w_t}^\top h_t$ — скалярное произведение для правильного слова. Мы хотим его максимизировать.
- Второе слагаемое $\log \sum_{w} \exp(v_w^\top h_t)$ — логарифм суммы экспонент. Мы хотим его минимизировать.

### 2.2 Правдоподобие для всего корпуса

Предполагая независимость позиций (при фиксированных параметрах), получаем:

$$
\mathcal{L}(U, V) = \prod_{t=1}^{T} P(w_t \mid C_t).
$$

**Логарифм правдоподобия:**

$$
\ell(U, V) = \log \mathcal{L} = \sum_{t=1}^{T} \log P(w_t \mid C_t).
$$

Подставляем выражение для $P$:

$$
\ell(U, V) = \sum_{t=1}^{T} \left[ v_{w_t}^\top h_t - \log \sum_{w \in V} \exp(v_w^\top h_t) \right].
$$

**Цель:** найти $U$ и $V$, максимизирующие $\ell$.

### 2.3 Свойства функции правдоподобия

**Невыпуклость.** Функция $\ell$ невыпукла по $U$ и $V$ из-за наличия softmax. Это означает, что глобальный максимум не гарантирован. На практике используется стохастический градиентный подъём (SGD), который сходится к локальному максимуму. Разные запуски с разной инициализацией могут давать разные результаты.

**Симметрия.** Если переставить слова в словаре, значение $\ell$ не изменится (при соответствующей перестановке строк $U$ и $V$). Это свойство идентифицируемости: решение не единственно. На практике это не проблема, потому что нас интересуют относительные расстояния между векторами, а не их абсолютные значения.

**Масштаб.** Если умножить все векторы на константу $c > 0$, значение $\ell$ изменится. Это означает, что нормы векторов не определены однозначно. На практике это не проблема, потому что нас интересует направление векторов, а не их длина. После обучения векторы часто нормализуют по L2.

---

## 3. Проблема вычислительной сложности softmax

### 3.1 Вычислительная стоимость

Знаменатель softmax:

$$
Z = \sum_{w \in V} \exp(v_w^\top h)
$$

требует суммирования по **всему словарю** $N$. Для каждого слова $w$ нужно:

1. Вычислить скалярное произведение $v_w^\top h$: $d$ умножений и $d-1$ сложений.
2. Вычислить экспоненту $\exp(v_w^\top h)$.

Итого $O(N \cdot d)$ операций на одну позицию.

**Оценка:** если $N = 10^6$, $d = 300$, $T = 10^9$ (корпус из миллиарда слов), то общее число операций:

$$
10^9 \times 10^6 \times 300 = 3 \times 10^{17}.
$$

Это невозможно даже для современных вычислительных кластеров. Для сравнения, современный GPU выполняет около $10^{13}$ операций в секунду. На вычисление $3 \times 10^{17}$ операций потребовалось бы $3 \times 10^4$ секунд, то есть около 8 часов, даже при идеальной параллелизации. А на практике операции не идеально параллельны, и время обучения было бы ещё больше.

### 3.2 Решения

Существуют три основных подхода к решению проблемы:

1. **Negative sampling:** заменить softmax на бинарную классификацию с $K$ отрицательными примерами. Сложность $O(K \cdot d)$ на позицию, где $K = 5$–$20$. Это уменьшает число операций в $N/K$ раз.

2. **Иерархический softmax:** заменить плоский softmax на дерево Хаффмана. Сложность $O(\log N \cdot d)$ на позицию. Это уменьшает число операций в $N / \log N$ раз.

3. **Субсэмплирование:** уменьшить число позиций, отбрасывая частые слова. Ускоряет обучение в 2–10 раз.

Рассмотрим каждый подход подробно.

---

## 4. CBOW с negative sampling

### 4.1 Идея negative sampling

Negative sampling заменяет задачу предсказания вероятности $P(w_t \mid C_t)$ через softmax по всему словарю на задачу **бинарной классификации**:

- **Положительный пример:** контекст $C_t$ и реальное целевое слово $w_t$. Метка: $D = 1$.
- **Отрицательные примеры:** контекст $C_t$ и $K$ случайных слов $w_{\text{neg}_1}, \ldots, w_{\text{neg}_K}$. Метка: $D = 0$.

Мы обучаем модель различать реальное целевое слово от случайных. Если модель хорошо справляется с этой задачей, значит, вектор контекста $h$ содержит достаточно информации, чтобы отличить правильное слово от неправильных.

**Ключевое преимущество:** вместо вычисления $N$ скалярных произведений (по одному на каждое слово словаря) мы вычисляем только $K+1$ скалярных произведений (одно для правильного слова и $K$ для отрицательных примеров). Поскольку $K \ll N$, это даёт огромное ускорение.

### 4.2 Вероятностная модель

Вероятность того, что слово $w$ является реальным целевым для контекста $C_t$:

$$
P(D = 1 \mid C_t, w) = \sigma(v_w^\top h_t),
$$

где $\sigma(x) = \frac{1}{1 + e^{-x}}$ — сигмоида, $h_t$ — усреднённый вектор контекста.

Разберём эту формулу. Сигмоида преобразует скалярное произведение (которое может быть любым числом от $-\infty$ до $+\infty$) в вероятность от 0 до 1. Если скалярное произведение велико и положительно, сигмоида близка к 1. Если велико и отрицательно, близка к 0.

Вероятность того, что слово $w$ является случайным:

$$
P(D = 0 \mid C_t, w) = 1 - \sigma(v_w^\top h_t) = \sigma(-v_w^\top h_t).
$$

**Свойство сигмоиды:** $\sigma(x) + \sigma(-x) = 1$. Это легко проверить:

$$
\sigma(x) + \sigma(-x) = \frac{1}{1+e^{-x}} + \frac{1}{1+e^{x}} = \frac{1+e^x + 1+e^{-x}}{(1+e^{-x})(1+e^x)} = \frac{2 + e^x + e^{-x}}{2 + e^x + e^{-x}} = 1.
$$

### 4.3 Функция потерь

Для одной позиции $t$ с $K$ отрицательными примерами:

$$
\mathcal{L}_t = \log \sigma(v_{w_t}^\top h_t) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top h_t).
$$

Разберём по частям.

**Первое слагаемое:** $\log \sigma(v_{w_t}^\top h_t)$. Это логарифм вероятности, что реальное целевое слово правильно классифицировано. Мы хотим его максимизировать. Поскольку $\log$ — монотонно возрастающая функция, максимизация $\log \sigma(x)$ эквивалентна максимизации $\sigma(x)$, что, в свою очередь, эквивалентно максимизации $x = v_{w_t}^\top h_t$.

**Второе слагаемое:** $\sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top h_t)$. Это сумма логарифмов вероятностей, что каждое из $K$ случайных слов правильно классифицировано как случайное. Мы тоже хотим максимизировать это слагаемое. Максимизация $\log \sigma(-x')$ эквивалентна максимизации $\sigma(-x')$, что эквивалентно минимизации $x' = v_{w_{\text{neg}}}^\top h_t$.

**Для всего корпуса:**

$$
\mathcal{L}(U, V) = \sum_{t=1}^{T} \left[ \log \sigma(v_{w_t}^\top h_t) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top h_t) \right].
$$

**Цель:** максимизировать $\mathcal{L}$ по $U$ и $V$.

### 4.4 Интуиция за функцией потерь

Рассмотрим, что происходит при максимизации каждого слагаемого.

**Первое слагаемое:** $\log \sigma(x)$, где $x = v_{w_t}^\top h_t$. Функция $\sigma(x)$ монотонно возрастает от 0 до 1. Логарифм $\log \sigma(x)$ тоже монотонно возрастает. Максимум достигается при $x \to \infty$. Это означает, что модель хочет сделать скалярное произведение $v_{w_t}^\top h_t$ как можно большим. Геометрически это означает, что выходной вектор реального целевого слова должен быть близок к вектору контекста.

**Второе слагаемое:** $\log \sigma(-x')$, где $x' = v_{w_{\text{neg}}}^\top h_t$. Функция $\sigma(-x')$ монотонно убывает по $x'$. Логарифм тоже монотонно убывает. Максимум достигается при $x' \to -\infty$. Это означает, что модель хочет сделать скалярное произведение $v_{w_{\text{neg}}}^\top h_t$ как можно меньшим. Геометрически это означает, что выходные векторы случайных слов должны быть далеки от вектора контекста.

Таким образом, обучение с negative sampling одновременно притягивает выходной вектор правильного слова к вектору контекста и отталкивает выходные векторы случайных слов от вектора контекста. Это создаёт структуру в пространстве эмбеддингов.

### 4.5 Выбор отрицательных примеров

Отрицательные примеры выбираются из **шумового распределения**:

$$
P_n(w) \propto \text{count}(w)^{3/4},
$$

где $\text{count}(w)$ — частота слова $w$ в корпусе.

**Нормированное распределение:**

$$
P_n(w) = \frac{\text{count}(w)^{3/4}}{\sum_{w' \in V} \text{count}(w')^{3/4}}.
$$

Почему именно степень 3/4? Рассмотрим три варианта.

**Вариант 1: униграммное распределение.** $P_n(w) \propto \text{count}(w)$. Частые слова выбираются очень часто. Например, слово «the», которое составляет 5% корпуса, будет выбираться как отрицательный пример в 5% случаев. Это означает, что модель будет тратить большую часть времени на различение реальных пар от пар с «the», что неэффективно. Редкие слова будут выбираться редко, и их векторы будут обучаться плохо.

**Вариант 2: равномерное распределение.** $P_n(w) = 1/N$. Все слова выбираются одинаково часто. Это означает, что редкие слова будут выбираться слишком часто. Модель будет тратить время на различение реальных пар от пар с редкими словами, которые и так встречаются редко. Частые слова, наоборот, будут выбираться недостаточно часто.

**Вариант 3: степень 3/4.** $P_n(w) \propto \text{count}(w)^{3/4}$. Это компромисс. Частые слова выбираются чаще, но не доминируют. Редкие слова тоже выбираются достаточно часто.

**Пример:** пусть слово $A$ встречается в 100 раз чаще слова $B$.

- Униграммное: $P_n(A)/P_n(B) = 100$.
- Степень 3/4: $P_n(A)/P_n(B) = 100^{3/4} \approx 31.6$.
- Равномерное: $P_n(A)/P_n(B) = 1$.

Степень 3/4 сглаживает распределение, делая его ближе к равномерному, но сохраняя предпочтение частым словам. Эмпирически было показано, что степень 3/4 даёт лучшее качество эмбеддингов, чем 1 или 0. Это одна из тех эвристик, которые хорошо работают на практике, хотя и не имеют строгого теоретического обоснования.

### 4.6 Количество отрицательных примеров $K$

**Рекомендации из оригинальной статьи:**

- Маленькие корпуса: $K = 5$–$20$.
- Большие корпуса: $K = 2$–$5$.

**Тонкий момент:** больше $K$ — точнее градиент (ближе к истинному softmax), но медленнее обучение. Меньше $K$ — быстрее, но шумнее.

**Эмпирическое правило:** $K = 5$–$10$ хорошо работает для большинства задач. При $K = 1$ обучение может быть нестабильным, потому что градиент слишком шумный. При $K > 20$ выигрыш в качестве становится незначительным, а время обучения растёт линейно.

---

## 5. Градиенты для CBOW с negative sampling

### 5.1 Обозначения

Для одной позиции $t$:

- $h = h_t \in \mathbb{R}^d$ — усреднённый вектор контекста;
- $v = v_{w_t} \in \mathbb{R}^d$ — выходной вектор реального целевого слова;
- $v'_k = v_{w_{\text{neg}_k}} \in \mathbb{R}^d$ — выходной вектор $k$-го отрицательного слова;
- $x = v^\top h$ — скалярное произведение для положительной пары;
- $x'_k = (v'_k)^\top h$ — скалярное произведение для $k$-й отрицательной пары;
- $c_1, \ldots, c_{2m}$ — контекстные слова.

### 5.2 Функция потерь

$$
\mathcal{L} = \log \sigma(x) + \sum_{k=1}^{K} \log \sigma(-x'_k).
$$

### 5.3 Производные сигмоиды

Вспомним:

$$
\sigma(x) = \frac{1}{1 + e^{-x}}.
$$

Производная:

$$
\sigma'(x) = \frac{d}{dx} \sigma(x) = \frac{e^{-x}}{(1 + e^{-x})^2} = \sigma(x)(1 - \sigma(x)).
$$

**Проверка:**

$$
\sigma(x)(1 - \sigma(x)) = \frac{1}{1+e^{-x}} \cdot \frac{e^{-x}}{1+e^{-x}} = \frac{e^{-x}}{(1+e^{-x})^2} = \sigma'(x).
$$

### 5.4 Производная логарифма сигмоиды

$$
\frac{\partial}{\partial x} \log \sigma(x) = \frac{\sigma'(x)}{\sigma(x)} = \frac{\sigma(x)(1 - \sigma(x))}{\sigma(x)} = 1 - \sigma(x).
$$

$$
\frac{\partial}{\partial x'} \log \sigma(-x') = \frac{-\sigma'(-x')}{\sigma(-x')} = -(1 - \sigma(-x')) = -\sigma(x').
$$

Здесь мы использовали $1 - \sigma(-x') = \sigma(x')$.

### 5.5 Градиент по выходному вектору $v$

$$
\frac{\partial \mathcal{L}}{\partial v} = \frac{\partial \mathcal{L}}{\partial x} \cdot \frac{\partial x}{\partial v} = (1 - \sigma(x)) \cdot h.
$$

Разберём:

- $\frac{\partial \mathcal{L}}{\partial x} = 1 - \sigma(x)$;
- $\frac{\partial x}{\partial v} = \frac{\partial (v^\top h)}{\partial v} = h$.

### 5.6 Градиент по выходному вектору $v'_k$

$$
\frac{\partial \mathcal{L}}{\partial v'_k} = \frac{\partial \mathcal{L}}{\partial x'_k} \cdot \frac{\partial x'_k}{\partial v'_k} = -\sigma(x'_k) \cdot h.
$$

### 5.7 Градиент по $h$

$$
\frac{\partial \mathcal{L}}{\partial h} = \frac{\partial \mathcal{L}}{\partial x} \cdot \frac{\partial x}{\partial h} + \sum_{k=1}^{K} \frac{\partial \mathcal{L}}{\partial x'_k} \cdot \frac{\partial x'_k}{\partial h}.
$$

Вычислим:

$$
\frac{\partial x}{\partial h} = \frac{\partial (v^\top h)}{\partial h} = v,
$$

$$
\frac{\partial x'_k}{\partial h} = \frac{\partial ((v'_k)^\top h)}{\partial h} = v'_k.
$$

Тогда:

$$
\frac{\partial \mathcal{L}}{\partial h} = (1 - \sigma(x)) v - \sum_{k=1}^{K} \sigma(x'_k) v'_k.
$$

### 5.8 Градиенты по входным векторам контекстных слов

Поскольку $h = \frac{1}{2m} \sum_{i=1}^{2m} u_{c_i}$, градиент по каждому $u_{c_i}$:

$$
\frac{\partial \mathcal{L}}{\partial u_{c_i}} = \frac{1}{2m} \cdot \frac{\partial \mathcal{L}}{\partial h} = \frac{1}{2m} \left[ (1 - \sigma(x)) v - \sum_{k=1}^{K} \sigma(x'_k) v'_k \right].
$$

**Тонкий момент:** все контекстные слова получают **одинаковое** обновление, потому что градиент по $h$ распределяется равномерно между ними. Это следствие усреднения. В Skip-gram каждое контекстное слово обновляется отдельно, потому что оно связано с целевым словом напрямую.

### 5.9 Обновление параметров

Используя градиентный подъём (мы максимизируем правдоподобие):

$$
v \leftarrow v + \eta (1 - \sigma(x)) h,
$$

$$
v'_k \leftarrow v'_k - \eta \sigma(x'_k) h, \quad k = 1, \ldots, K,
$$

$$
u_{c_i} \leftarrow u_{c_i} + \frac{\eta}{2m} \left[ (1 - \sigma(x)) v - \sum_{k=1}^{K} \sigma(x'_k) v'_k \right], \quad i = 1, \ldots, 2m.
$$

### 5.10 Интерпретация обновлений

**Выходной вектор $v$ целевого слова:** обновляется в направлении $h$ с коэффициентом $(1 - \sigma(x))$. Если модель уже уверена, что слово правильно ($\sigma(x) \approx 1$), коэффициент близок к нулю, и обновление слабое. Если модель ошибается ($\sigma(x) \approx 0$), коэффициент близок к единице, и обновление сильное. Это означает, что модель учится больше на тех примерах, где она ошибается.

**Выходные векторы $v'_k$ отрицательных слов:** обновляются в направлении $-h$ с коэффициентом $\sigma(x'_k)$. Если модель ошибочно считает отрицательное слово правильным ($\sigma(x'_k) \approx 1$), коэффициент близок к единице, и вектор $v'_k$ сильно отталкивается от $h$. Если модель правильно считает пару случайной ($\sigma(x'_k) \approx 0$), обновление слабое.

**Входные векторы $u_{c_i}$ контекстных слов:** обновляются в направлении $(1 - \sigma(x)) v - \sum_k \sigma(x'_k) v'_k$, делённом на $2m$. Это означает, что контекстные слова притягиваются к реальному целевому слову и отталкиваются от отрицательных. Все контекстные слова получают одинаковое обновление, потому что они вносят одинаковый вклад в вектор $h$.

---

## 6. CBOW с иерархическим softmax

### 6.1 Идея

Иерархический softmax использует бинарное дерево (дерево Хаффмана), в листьях которого находятся слова. Вероятность слова вычисляется как произведение вероятностей на пути от корня до листа.

**Преимущество:** сложность $O(\log N)$ вместо $O(N)$.

### 6.2 Построение дерева Хаффмана

1. Каждое слово — лист дерева. Вес листа — частота слова в корпусе.
2. На каждом шаге выбираются два узла с наименьшими весами и объединяются в новый узел с суммой весов.
3. Процесс повторяется, пока не останется один корневой узел.

**Результат:** частые слова ближе к корню (короткие пути), редкие — дальше (длинные пути).

**Пример:** словарь $\{\text{кошка}, \text{сидит}, \text{на}, \text{окне}\}$ с частотами $\{5, 4, 3, 2\}$.

Шаг 1: объединяем «окне» (2) и «на» (3) → узел A с весом 5.
Шаг 2: объединяем «сидит» (4) и A (5) → узел B с весом 9.
Шаг 3: объединяем «кошка» (5) и B (9) → корень с весом 14.

Дерево:

```
        корень
       /      \
    кошка      B
             /   \
          сидит   A
                /   \
              на    окне
```

Пути:

- кошка: 1 шаг (корень → кошка).
- сидит: 2 шага (корень → B → сидит).
- на: 3 шага (корень → B → A → на).
- окне: 3 шага (корень → B → A → окне).

### 6.3 Формула вероятности

Пусть путь от корня до слова $w$ состоит из узлов $n_1, \ldots, n_L$. На каждом узле $n_i$ есть бинарный выбор $d_i \in \{0, 1\}$ (0 — налево, 1 — направо).

Вероятность слова $w$:

$$
P(w \mid C_t) = \prod_{i=1}^{L} \sigma\left( (-1)^{d_i} v_{n_i}^\top h_t \right).
$$

**Разберём:**

- Если $d_i = 0$ (налево): $(-1)^{d_i} = 1$, вероятность $\sigma(v_{n_i}^\top h_t)$.
- Если $d_i = 1$ (направо): $(-1)^{d_i} = -1$, вероятность $\sigma(-v_{n_i}^\top h_t)$.

**Интуиция:** на каждом узле мы делаем бинарный выбор. Вероятность слова — это произведение вероятностей всех выборов на пути. Это похоже на то, как мы принимаем последовательность бинарных решений, чтобы добраться до нужного слова.

### 6.4 Функция потерь

$$
\mathcal{L} = \sum_{t=1}^{T} \sum_{i=1}^{L_t} \log \sigma\left( (-1)^{d_i^t} v_{n_i^t}^\top h_t \right).
$$

### 6.5 Градиенты

Для каждого узла $n_i$ на пути:

$$
z_i = (-1)^{d_i} v_{n_i}^\top h_t.
$$

Градиент по $v_{n_i}$:

$$
\frac{\partial \mathcal{L}}{\partial v_{n_i}} = (-1)^{d_i} (1 - \sigma(z_i)) h_t.
$$

Градиент по $h_t$:

$$
\frac{\partial \mathcal{L}}{\partial h_t} = \sum_{i=1}^{L_t} (-1)^{d_i} (1 - \sigma(z_i)) v_{n_i}.
$$

Градиенты по входным векторам контекстных слов:

$$
\frac{\partial \mathcal{L}}{\partial u_{c_j}} = \frac{1}{2m} \frac{\partial \mathcal{L}}{\partial h_t}.
$$

### 6.6 Сложность

- Плоский softmax: $O(N)$.
- Иерархический softmax: $O(\log N)$.

**Пример:** $N = 10^6$, $\log_2 N \approx 20$. Ускорение в 50 000 раз.

---

## 7. Субсэмплирование частых слов

### 7.1 Проблема

Частые слова (артикли, предлоги, союзы) встречаются в корпусе миллионы раз. Они дают много обучающих пар, но не несут семантической информации. Более того, они доминируют в обучении, что замедляет сходимость и ухудшает качество векторов для редких слов.

### 7.2 Формула

Вероятность **удалить** слово $w$:

$$
P_{\text{discard}}(w) = 1 - \sqrt{\frac{t}{f(w)}},
$$

где $f(w)$ — частота слова в корпусе, $t$ — порог (обычно $t = 10^{-5}$).

### 7.3 Анализ

- Если $f(w) < t$: $P_{\text{discard}} < 0$, слово всегда сохраняется.
- Если $f(w) = t$: $P_{\text{discard}} = 0$, слово сохраняется.
- Если $f(w) > t$: $P_{\text{discard}} > 0$, слово удаляется с некоторой вероятностью.
- Если $f(w) \gg t$: $P_{\text{discard}} \approx 1$, слово удаляется почти всегда.

### 7.4 Пример

$t = 10^{-5}$.

- Слово «the»: $f = 0.05$. $P_{\text{discard}} = 1 - \sqrt{10^{-5}/0.05} = 1 - \sqrt{0.0002} \approx 0.986$. Удаляется с вероятностью 98.6%.
- Слово «кошка»: $f = 10^{-5}$. $P_{\text{discard}} = 1 - \sqrt{10^{-5}/10^{-5}} = 0$. Сохраняется.
- Слово «диван»: $f = 10^{-6}$. $P_{\text{discard}} = 1 - \sqrt{10^{-5}/10^{-6}} = 1 - \sqrt{10} < 0$. Сохраняется.

### 7.5 Влияние

- Ускоряет обучение в 2–10 раз.
- Улучшает качество векторов для редких слов.
- Удаляет шум от частых слов.

---

## 8. Практические рекомендации

### 8.1 Размерность $d$

- $d = 50$: маленькие корпуса, быстрые эксперименты.
- $d = 100$: стандарт для большинства задач.
- $d = 300$: оригинальная статья Word2Vec.
- $d = 500$–$1000$: очень большие корпуса, сложные задачи.

### 8.2 Размер окна $m$

- $m = 2$: синтаксическая близость.
- $m = 5$: семантическая близость.
- $m = 10$: тематическая близость.

### 8.3 Динамическое окно

Для каждой позиции размер окна выбирается случайно от 1 до $m$. Это даёт больше веса близким словам.

### 8.4 Инициализация

Случайные малые значения из $[-0.5/d, 0.5/d]$.

### 8.5 Скорость обучения

$\eta_0 = 0.025$, линейно убывает до $\eta_{\min} = 10^{-4}$.

### 8.6 Количество эпох

3–5 эпох.

### 8.7 Нормализация

После обучения векторы нормализуют по L2.

### 8.8 Оценка качества

- Внутренняя: аналогии, близость слов.
- Внешняя: downstream-задачи.

---

## 9. Сравнение CBOW и Skip-gram

| Свойство | CBOW | Skip-gram |
|----------|------|-----------|
| Задача | По контексту предсказать слово | По слову предсказать контекст |
| Вход | Несколько слов | Одно слово |
| Выход | Одно слово | Несколько слов |
| Число пар на позицию | 1 | $2m$ |
| Скорость | Быстрее | Медленнее |
| Качество для редких слов | Хуже | Лучше |
| Качество для частых слов | Лучше | Хуже |
| Память | Меньше | Больше |

**Тонкий момент:** CBOW быстрее в $2m$ раз, но Skip-gram даёт лучшее качество для редких слов. На практике выбор зависит от задачи.

---

## 10. Заключение

CBOW — это архитектура Word2Vec, которая предсказывает целевое слово по контексту. Она использует усреднение контекстных векторов, что делает её быстрой и устойчивой к шуму. CBOW обучается через negative sampling или иерархический softmax, что позволяет избежать softmax по всему словарю.

**Ключевые формулы:**

Вероятность целевого слова:

$$
P(w_t \mid C_t) = \frac{\exp(v_{w_t}^\top h_t)}{\sum_{w \in V} \exp(v_w^\top h_t)},
$$

где $h_t = \frac{1}{2m} \sum_{c \in C_t} u_c$.

Функция потерь с negative sampling:

$$
\mathcal{L} = \sum_{t=1}^{T} \left[ \log \sigma(v_{w_t}^\top h_t) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top h_t) \right].
$$

Градиенты:

$$
\frac{\partial \mathcal{L}}{\partial v_{w_t}} = (1 - \sigma(x)) h_t,
$$

$$
\frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}_k}}} = -\sigma(x'_k) h_t,
$$

$$
\frac{\partial \mathcal{L}}{\partial u_{c_i}} = \frac{1}{2m} \left[ (1 - \sigma(x)) v_{w_t} - \sum_{k=1}^{K} \sigma(x'_k) v_{w_{\text{neg}_k}} \right].
$$

Эти формулы — основа CBOW. Их понимание позволяет эффективно обучать эмбеддинги и применять их в реальных задачах.

---

**В следующей части** мы подробно разберём **Skip-gram** с полным выводом градиентов и численным примером на учебном корпусе.

# Численный пример CBOW на учебном корпусе

## 1. Постановка задачи

Рассмотрим тот же учебный корпус из трёх документов:

- $d_1$: «кошка сидит на окне»
- $d_2$: «собака сидит на крыльце»
- $d_3$: «кошка спит на диване»

Объединим документы в одну последовательность (игнорируя границы):

$$
\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{сидит}, \text{на}, \text{крыльце}, \text{кошка}, \text{спит}, \text{на}, \text{диване}.
$$

Длина последовательности $T = 12$. Словарь:

$$
V = \{\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване}\},
$$

размер словаря $N = 8$.

**Параметры:**

- окно $m = 1$ (для простоты);
- размерность эмбеддинга $d = 2$ (для визуализации);
- $K = 1$ отрицательный пример;
- скорость обучения $\eta = 0.1$.

**Тонкий момент:** в реальных задачах $d = 100$–$300$, $m = 5$–$10$, $K = 5$–$20$. Мы используем маленькие значения, чтобы вычисления были обозримыми и можно было проверить каждый шаг вручную.

## 2. Генерация обучающих пар

Для каждой позиции $t$ целевое слово — это $w_t$, а контекстные слова — это $w_{t-1}$ и $w_{t+1}$ (поскольку $m = 1$).

**Ключевое отличие от Skip-gram:** в CBOW для каждой позиции генерируется **одна** обучающая пара (контекст → целевое слово). В Skip-gram генерируется **две** пары (целевое слово → каждое контекстное слово).

| Позиция $t$ | Целевое $w_t$ | Контекст $C_t$ | Обучающая пара CBOW |
|-------------|---------------|----------------|---------------------|
| 1 | кошка | $\{$сидит$\}$ | ($\{$сидит$\}$, кошка) |
| 2 | сидит | $\{$кошка, на$\}$ | ($\{$кошка, на$\}$, сидит) |
| 3 | на | $\{$сидит, окне$\}$ | ($\{$сидит, окне$\}$, на) |
| 4 | окне | $\{$на, собака$\}$ | ($\{$на, собака$\}$, окне) |
| 5 | собака | $\{$окне, сидит$\}$ | ($\{$окне, сидит$\}$, собака) |
| 6 | сидит | $\{$собака, на$\}$ | ($\{$собака, на$\}$, сидит) |
| 7 | на | $\{$сидит, крыльце$\}$ | ($\{$сидит, крыльце$\}$, на) |
| 8 | крыльце | $\{$на, кошка$\}$ | ($\{$на, кошка$\}$, крыльце) |
| 9 | кошка | $\{$крыльце, спит$\}$ | ($\{$крыльце, спит$\}$, кошка) |
| 10 | спит | $\{$кошка, на$\}$ | ($\{$кошка, на$\}$, спит) |
| 11 | на | $\{$спит, диване$\}$ | ($\{$спит, диване$\}$, на) |
| 12 | диване | $\{$на$\}$ | ($\{$на$\}$, диване) |

**Итого:** $12$ обучающих пар — по одной на каждую позицию. В Skip-gram было бы $23$ пары.

**Важно:** в CBOW контекст — это **множество** слов, порядок не важен. Мы просто усредняем их векторы.

## 3. Инициализация параметров

Зададим начальные векторы для каждого слова. Для простоты используем конкретные значения.

**Входные векторы $U$ (используются для контекстных слов):**

| Слово | $u_w$ |
|-------|-------|
| кошка | $(0.2, -0.1)$ |
| сидит | $(0.3, 0.4)$ |
| на | $(-0.1, 0.6)$ |
| окне | $(0.5, -0.3)$ |
| собака | $(0.1, 0.2)$ |
| крыльце | $(-0.4, 0.1)$ |
| спит | $(0.6, 0.5)$ |
| диване | $(-0.2, -0.5)$ |

**Выходные векторы $V$ (используются для целевого слова):**

| Слово | $v_w$ |
|-------|-------|
| кошка | $(0.1, 0.3)$ |
| сидит | $(-0.2, 0.4)$ |
| на | $(0.5, -0.1)$ |
| окне | $(0.3, 0.2)$ |
| собака | $(-0.3, -0.2)$ |
| крыльце | $(0.4, 0.5)$ |
| спит | $(-0.1, 0.6)$ |
| диване | $(0.2, -0.4)$ |

**Ключевое отличие от Skip-gram:** в CBOW входные векторы используются для **контекстных** слов, а выходные — для **целевого** слова. В Skip-gram наоборот: входные — для целевого, выходные — для контекстных.

## 4. Формулы для одного шага обучения

Для одной обучающей пары (контекст $C$, целевое слово $w_t$) с одним отрицательным примером $w_{\text{neg}}$:

### 4.1 Прямой проход

**Шаг 1: усреднение контекстных векторов**

$$
h = \frac{1}{|C|} \sum_{c \in C} u_c = \frac{1}{2m} \sum_{i=1}^{2m} u_{c_i}.
$$

**Шаг 2: скалярное произведение для положительной пары**

$$
x = v_{w_t}^\top h = \sum_{k=1}^{d} v_{w_t, k} \cdot h_k.
$$

**Шаг 3: сигмоида**

$$
\sigma(x) = \frac{1}{1 + e^{-x}}.
$$

**Шаг 4: скалярное произведение для отрицательной пары**

$$
x' = v_{w_{\text{neg}}}^\top h = \sum_{k=1}^{d} v_{w_{\text{neg}}, k} \cdot h_k.
$$

**Шаг 5: сигмоида**

$$
\sigma(x') = \frac{1}{1 + e^{-x'}}.
$$

**Шаг 6: функция потерь**

$$
\mathcal{L} = \log \sigma(x) + \log \sigma(-x').
$$

### 4.2 Обратный проход (градиенты)

**Градиент по выходному вектору целевого слова:**

$$
\frac{\partial \mathcal{L}}{\partial v_{w_t}} = (1 - \sigma(x)) \cdot h.
$$

**Градиент по выходному вектору отрицательного слова:**

$$
\frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}}}} = -\sigma(x') \cdot h.
$$

**Градиент по $h$:**

$$
\frac{\partial \mathcal{L}}{\partial h} = (1 - \sigma(x)) \cdot v_{w_t} - \sigma(x') \cdot v_{w_{\text{neg}}}.
$$

**Градиент по каждому входному вектору контекстного слова:**

$$
\frac{\partial \mathcal{L}}{\partial u_{c_i}} = \frac{1}{2m} \cdot \frac{\partial \mathcal{L}}{\partial h}, \quad i = 1, \ldots, 2m.
$$

**Тонкий момент:** все контекстные слова получают **одинаковое** обновление, потому что градиент по $h$ распределяется равномерно между ними. Это следствие усреднения.

### 4.3 Обновление параметров (градиентный подъём)

$$
v_{w_t} \leftarrow v_{w_t} + \eta \cdot \frac{\partial \mathcal{L}}{\partial v_{w_t}},
$$

$$
v_{w_{\text{neg}}} \leftarrow v_{w_{\text{neg}}} + \eta \cdot \frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}}}},
$$

$$
u_{c_i} \leftarrow u_{c_i} + \eta \cdot \frac{\partial \mathcal{L}}{\partial u_{c_i}}, \quad i = 1, \ldots, 2m.
$$

## 5. Первая пара: контекст $\{$кошка, на$\}$, целевое «сидит»

Возьмём позицию $t = 2$: целевое слово «сидит», контекст $\{$кошка, на$\}$. Выберем отрицательный пример «диване».

### 5.1 Прямой проход

**Шаг 1: усреднение контекстных векторов.**

$$
u_{\text{кошка}} = (0.2, -0.1), \quad u_{\text{на}} = (-0.1, 0.6).
$$

$$
h = \frac{1}{2} \left[ (0.2, -0.1) + (-0.1, 0.6) \right] = \frac{1}{2} (0.1, 0.5) = (0.05, 0.25).
$$

**Шаг 2: скалярное произведение для положительной пары.**

$$
v_{\text{сидит}} = (-0.2, 0.4).
$$

$$
x = v_{\text{сидит}}^\top h = (-0.2) \cdot 0.05 + 0.4 \cdot 0.25 = -0.01 + 0.10 = 0.09.
$$

**Шаг 3: сигмоида.**

$$
\sigma(x) = \sigma(0.09) = \frac{1}{1 + e^{-0.09}} = \frac{1}{1 + 0.9139} = \frac{1}{1.9139} \approx 0.5225.
$$

**Шаг 4: скалярное произведение для отрицательной пары.**

$$
v_{\text{диване}} = (0.2, -0.4).
$$

$$
x' = v_{\text{диване}}^\top h = 0.2 \cdot 0.05 + (-0.4) \cdot 0.25 = 0.01 - 0.10 = -0.09.
$$

**Шаг 5: сигмоида.**

$$
\sigma(x') = \sigma(-0.09) = \frac{1}{1 + e^{0.09}} = \frac{1}{1 + 1.0942} = \frac{1}{2.0942} \approx 0.4775.
$$

**Шаг 6: функция потерь.**

$$
\mathcal{L} = \log(0.5225) + \log(0.5225) = 2 \cdot (-0.6489) = -1.2978.
$$

**Тонкий момент:** обратите внимание, что $\sigma(x) = 0.5225$ и $\sigma(-x') = 0.5225$, потому что $x' = -x = -0.09$. Это совпадение из-за симметрии начальных векторов.

### 5.2 Обратный проход

**Градиент по $v_{\text{сидит}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{сидит}}} = (1 - \sigma(x)) \cdot h = (1 - 0.5225) \cdot (0.05, 0.25) = 0.4775 \cdot (0.05, 0.25) = (0.0239, 0.1194).
$$

**Градиент по $v_{\text{диване}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{диване}}} = -\sigma(x') \cdot h = -0.4775 \cdot (0.05, 0.25) = (-0.0239, -0.1194).
$$

**Градиент по $h$:**

$$
\frac{\partial \mathcal{L}}{\partial h} = (1 - \sigma(x)) \cdot v_{\text{сидит}} - \sigma(x') \cdot v_{\text{диване}}.
$$

Вычислим первое слагаемое:

$$
(1 - \sigma(x)) \cdot v_{\text{сидит}} = 0.4775 \cdot (-0.2, 0.4) = (-0.0955, 0.1910).
$$

Вычислим второе слагаемое:

$$
\sigma(x') \cdot v_{\text{диване}} = 0.4775 \cdot (0.2, -0.4) = (0.0955, -0.1910).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial h} = (-0.0955, 0.1910) - (0.0955, -0.1910) = (-0.1910, 0.3820).
$$

**Градиенты по входным векторам контекстных слов:**

Поскольку $h = \frac{1}{2}(u_{\text{кошка}} + u_{\text{на}})$, каждый контекстный вектор получает градиент:

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{кошка}}} = \frac{1}{2} \cdot \frac{\partial \mathcal{L}}{\partial h} = \frac{1}{2} (-0.1910, 0.3820) = (-0.0955, 0.1910).
$$

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{на}}} = \frac{1}{2} \cdot \frac{\partial \mathcal{L}}{\partial h} = (-0.0955, 0.1910).
$$

### 5.3 Обновление параметров

**Обновляем $v_{\text{сидит}}$:**

$$
v_{\text{сидит}} \leftarrow (-0.2, 0.4) + 0.1 \cdot (0.0239, 0.1194) = (-0.2 + 0.0024, 0.4 + 0.0119) = (-0.1976, 0.4119).
$$

**Обновляем $v_{\text{диване}}$:**

$$
v_{\text{диване}} \leftarrow (0.2, -0.4) + 0.1 \cdot (-0.0239, -0.1194) = (0.2 - 0.0024, -0.4 - 0.0119) = (0.1976, -0.4119).
$$

**Обновляем $u_{\text{кошка}}$:**

$$
u_{\text{кошка}} \leftarrow (0.2, -0.1) + 0.1 \cdot (-0.0955, 0.1910) = (0.2 - 0.0096, -0.1 + 0.0191) = (0.1904, -0.0809).
$$

**Обновляем $u_{\text{на}}$:**

$$
u_{\text{на}} \leftarrow (-0.1, 0.6) + 0.1 \cdot (-0.0955, 0.1910) = (-0.1 - 0.0096, 0.6 + 0.0191) = (-0.1096, 0.6191).
$$

**Интерпретация обновлений:**

- Вектор $v_{\text{сидит}}$ сдвинулся в направлении $h = (0.05, 0.25)$. Это значит, что выходной вектор целевого слова стал ближе к усреднённому контексту.
- Вектор $v_{\text{диване}}$ сдвинулся в направлении $-h$. Он оттолкнулся от контекста.
- Векторы $u_{\text{кошка}}$ и $u_{\text{на}}$ получили **одинаковое** обновление, потому что они одинаково вносят вклад в $h$.

## 6. Вторая пара: контекст $\{$сидит, окне$\}$, целевое «на»

Перейдём к позиции $t = 3$: целевое слово «на», контекст $\{$сидит, окне$\}$. Выберем отрицательный пример «собака».

**Важно:** мы используем **обновлённые** векторы из предыдущего шага. В частности, $u_{\text{сидит}}$ ещё не обновлялся (он был выходным в первой паре), а $u_{\text{на}}$ обновлён до $(-0.1096, 0.6191)$.

### 6.1 Прямой проход

**Шаг 1: усреднение контекстных векторов.**

$$
u_{\text{сидит}} = (0.3, 0.4), \quad u_{\text{окне}} = (0.5, -0.3).
$$

$$
h = \frac{1}{2} \left[ (0.3, 0.4) + (0.5, -0.3) \right] = \frac{1}{2} (0.8, 0.1) = (0.4, 0.05).
$$

**Шаг 2: скалярное произведение для положительной пары.**

$$
v_{\text{на}} = (0.5, -0.1).
$$

$$
x = v_{\text{на}}^\top h = 0.5 \cdot 0.4 + (-0.1) \cdot 0.05 = 0.20 - 0.005 = 0.195.
$$

**Шаг 3: сигмоида.**

$$
\sigma(x) = \sigma(0.195) = \frac{1}{1 + e^{-0.195}} = \frac{1}{1 + 0.8228} = \frac{1}{1.8228} \approx 0.5486.
$$

**Шаг 4: скалярное произведение для отрицательной пары.**

$$
v_{\text{собака}} = (-0.3, -0.2).
$$

$$
x' = v_{\text{собака}}^\top h = (-0.3) \cdot 0.4 + (-0.2) \cdot 0.05 = -0.12 - 0.01 = -0.13.
$$

**Шаг 5: сигмоида.**

$$
\sigma(x') = \sigma(-0.13) = \frac{1}{1 + e^{0.13}} = \frac{1}{1 + 1.1388} = \frac{1}{2.1388} \approx 0.4675.
$$

**Шаг 6: функция потерь.**

$$
\mathcal{L} = \log(0.5486) + \log(0.5325) \approx -0.6003 - 0.6300 = -1.2303.
$$

### 6.2 Обратный проход

**Градиент по $v_{\text{на}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{на}}} = (1 - \sigma(x)) \cdot h = (1 - 0.5486) \cdot (0.4, 0.05) = 0.4514 \cdot (0.4, 0.05) = (0.1806, 0.0226).
$$

**Градиент по $v_{\text{собака}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{собака}}} = -\sigma(x') \cdot h = -0.4675 \cdot (0.4, 0.05) = (-0.1870, -0.0234).
$$

**Градиент по $h$:**

$$
\frac{\partial \mathcal{L}}{\partial h} = 0.4514 \cdot (0.5, -0.1) - 0.4675 \cdot (-0.3, -0.2).
$$

Первое слагаемое:

$$
0.4514 \cdot (0.5, -0.1) = (0.2257, -0.0451).
$$

Второе слагаемое:

$$
0.4675 \cdot (-0.3, -0.2) = (-0.1403, -0.0935).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial h} = (0.2257, -0.0451) - (-0.1403, -0.0935) = (0.3660, 0.0484).
$$

**Градиенты по входным векторам контекстных слов:**

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{сидит}}} = \frac{1}{2} (0.3660, 0.0484) = (0.1830, 0.0242).
$$

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{окне}}} = (0.1830, 0.0242).
$$

### 6.3 Обновление параметров

**Обновляем $v_{\text{на}}$:**

$$
v_{\text{на}} \leftarrow (0.5, -0.1) + 0.1 \cdot (0.1806, 0.0226) = (0.5181, -0.0977).
$$

**Обновляем $v_{\text{собака}}$:**

$$
v_{\text{собака}} \leftarrow (-0.3, -0.2) + 0.1 \cdot (-0.1870, -0.0234) = (-0.3187, -0.2023).
$$

**Обновляем $u_{\text{сидит}}$:**

$$
u_{\text{сидит}} \leftarrow (0.3, 0.4) + 0.1 \cdot (0.1830, 0.0242) = (0.3183, 0.4024).
$$

**Обновляем $u_{\text{окне}}$:**

$$
u_{\text{окне}} \leftarrow (0.5, -0.3) + 0.1 \cdot (0.1830, 0.0242) = (0.5183, -0.2976).
$$

**Наблюдение:** вектор $u_{\text{сидит}}$ обновился впервые (в первой паре он был выходным). Он сдвинулся в направлении, которое делает его более совместимым с контекстом $\{$сидит, окне$\}$ для предсказания «на».

## 7. Третья пара: контекст $\{$на, собака$\}$, целевое «окне»

Перейдём к позиции $t = 4$: целевое слово «окне», контекст $\{$на, собака$\}$. Выберем отрицательный пример «спит».

**Важно:** используем обновлённые векторы. $u_{\text{на}} = (-0.1096, 0.6191)$, $u_{\text{собака}} = (0.1, 0.2)$ (ещё не обновлялся).

### 7.1 Прямой проход

**Шаг 1: усреднение.**

$$
h = \frac{1}{2} \left[ (-0.1096, 0.6191) + (0.1, 0.2) \right] = \frac{1}{2} (-0.0096, 0.8191) = (-0.0048, 0.4096).
$$

**Шаг 2: скалярное произведение для положительной пары.**

$$
v_{\text{окне}} = (0.3, 0.2).
$$

$$
x = 0.3 \cdot (-0.0048) + 0.2 \cdot 0.4096 = -0.0014 + 0.0819 = 0.0805.
$$

**Шаг 3: сигмоида.**

$$
\sigma(x) = \sigma(0.0805) = \frac{1}{1 + e^{-0.0805}} \approx \frac{1}{1 + 0.9227} = \frac{1}{1.9227} \approx 0.5201.
$$

**Шаг 4: скалярное произведение для отрицательной пары.**

$$
v_{\text{спит}} = (-0.1, 0.6).
$$

$$
x' = (-0.1) \cdot (-0.0048) + 0.6 \cdot 0.4096 = 0.0005 + 0.2458 = 0.2463.
$$

**Шаг 5: сигмоида.**

$$
\sigma(x') = \sigma(0.2463) = \frac{1}{1 + e^{-0.2463}} \approx \frac{1}{1 + 0.7816} = \frac{1}{1.7816} \approx 0.5613.
$$

**Шаг 6: функция потерь.**

$$
\mathcal{L} = \log(0.5201) + \log(0.4387) \approx -0.6537 - 0.8239 = -1.4776.
$$

**Наблюдение:** функция потерь больше по модулю, чем в предыдущих парах, потому что отрицательный пример «спит» имеет высокую скалярную произведению с $h$, что означает, что модель ошибочно считает его совместимым с контекстом.

### 7.2 Обратный проход

**Градиент по $v_{\text{окне}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{окне}}} = (1 - 0.5201) \cdot (-0.0048, 0.4096) = 0.4799 \cdot (-0.0048, 0.4096) = (-0.0023, 0.1966).
$$

**Градиент по $v_{\text{спит}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{спит}}} = -0.5613 \cdot (-0.0048, 0.4096) = (0.0027, -0.2299).
$$

**Градиент по $h$:**

$$
\frac{\partial \mathcal{L}}{\partial h} = 0.4799 \cdot (0.3, 0.2) - 0.5613 \cdot (-0.1, 0.6).
$$

Первое слагаемое:

$$
0.4799 \cdot (0.3, 0.2) = (0.1440, 0.0960).
$$

Второе слагаемое:

$$
0.5613 \cdot (-0.1, 0.6) = (-0.0561, 0.3368).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial h} = (0.1440, 0.0960) - (-0.0561, 0.3368) = (0.2001, -0.2408).
$$

**Градиенты по входным векторам:**

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{на}}} = \frac{1}{2} (0.2001, -0.2408) = (0.1001, -0.1204).
$$

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{собака}}} = (0.1001, -0.1204).
$$

### 7.3 Обновление

**Обновляем $v_{\text{окне}}$:**

$$
v_{\text{окне}} \leftarrow (0.3, 0.2) + 0.1 \cdot (-0.0023, 0.1966) = (0.2998, 0.2197).
$$

**Обновляем $v_{\text{спит}}$:**

$$
v_{\text{спит}} \leftarrow (-0.1, 0.6) + 0.1 \cdot (0.0027, -0.2299) = (-0.0997, 0.5770).
$$

**Обновляем $u_{\text{на}}$:**

$$
u_{\text{на}} \leftarrow (-0.1096, 0.6191) + 0.1 \cdot (0.1001, -0.1204) = (-0.0996, 0.6071).
$$

**Обновляем $u_{\text{собака}}$:**

$$
u_{\text{собака}} \leftarrow (0.1, 0.2) + 0.1 \cdot (0.1001, -0.1204) = (0.1100, 0.1880).
$$

## 8. Сводка обновлений после первых трёх пар

| Вектор | Начальное значение | После пары 1 | После пары 2 | После пары 3 |
|--------|-------------------|--------------|--------------|--------------|
| $u_{\text{кошка}}$ | $(0.2, -0.1)$ | $(0.1904, -0.0809)$ | $(0.1904, -0.0809)$ | $(0.1904, -0.0809)$ |
| $u_{\text{на}}$ | $(-0.1, 0.6)$ | $(-0.1096, 0.6191)$ | $(-0.1096, 0.6191)$ | $(-0.0996, 0.6071)$ |
| $u_{\text{сидит}}$ | $(0.3, 0.4)$ | $(0.3, 0.4)$ | $(0.3183, 0.4024)$ | $(0.3183, 0.4024)$ |
| $u_{\text{окне}}$ | $(0.5, -0.3)$ | $(0.5, -0.3)$ | $(0.5183, -0.2976)$ | $(0.5183, -0.2976)$ |
| $u_{\text{собака}}$ | $(0.1, 0.2)$ | $(0.1, 0.2)$ | $(0.1, 0.2)$ | $(0.1100, 0.1880)$ |
| $v_{\text{сидит}}$ | $(-0.2, 0.4)$ | $(-0.1976, 0.4119)$ | $(-0.1976, 0.4119)$ | $(-0.1976, 0.4119)$ |
| $v_{\text{диване}}$ | $(0.2, -0.4)$ | $(0.1976, -0.4119)$ | $(0.1976, -0.4119)$ | $(0.1976, -0.4119)$ |
| $v_{\text{на}}$ | $(0.5, -0.1)$ | $(0.5, -0.1)$ | $(0.5181, -0.0977)$ | $(0.5181, -0.0977)$ |
| $v_{\text{собака}}$ | $(-0.3, -0.2)$ | $(-0.3, -0.2)$ | $(-0.3187, -0.2023)$ | $(-0.3187, -0.2023)$ |
| $v_{\text{окне}}$ | $(0.3, 0.2)$ | $(0.3, 0.2)$ | $(0.3, 0.2)$ | $(0.2998, 0.2197)$ |
| $v_{\text{спит}}$ | $(-0.1, 0.6)$ | $(-0.1, 0.6)$ | $(-0.1, 0.6)$ | $(-0.0997, 0.5770)$ |

**Наблюдения:**

- Вектор $u_{\text{на}}$ обновился дважды: сначала как контекстное слово в паре 1, затем как контекстное слово в паре 3. Он сдвинулся.
- Вектор $u_{\text{сидит}}$ обновился один раз: как контекстное слово в паре 2.
- Вектор $u_{\text{окне}}$ обновился один раз: как контекстное слово в паре 2.
- Вектор $u_{\text{собака}}$ обновился один раз: как контекстное слово в паре 3.
- Выходные векторы обновлялись в тех парах, где они были целевыми.

## 9. Вероятностная интерпретация

Рассмотрим, как меняется вероятность $P(w_t \mid C)$ для пары 1 (контекст $\{$кошка, на$\}$, целевое «сидит») до и после обновления.

**До обучения:**

$$
x = v_{\text{сидит}}^\top h = 0.09, \quad \sigma(x) \approx 0.5225.
$$

**После обновления (пары 1):**

Мы обновили $v_{\text{сидит}}$ до $(-0.1976, 0.4119)$ и $h$ (через $u_{\text{кошка}}$ и $u_{\text{на}}$):

$$
u_{\text{кошка}} = (0.1904, -0.0809), \quad u_{\text{на}} = (-0.1096, 0.6191).
$$

$$
h_{\text{new}} = \frac{1}{2} \left[ (0.1904, -0.0809) + (-0.1096, 0.6191) \right] = \frac{1}{2} (0.0808, 0.5382) = (0.0404, 0.2691).
$$

$$
x_{\text{new}} = (-0.1976) \cdot 0.0404 + 0.4119 \cdot 0.2691 = -0.0080 + 0.1108 = 0.1028.
$$

$$
\sigma(x_{\text{new}}) = \sigma(0.1028) \approx 0.5257.
$$

**Наблюдение:** вероятность выросла с 0.5225 до 0.5257. Это означает, что модель стала немного более уверена в правильности целевого слова. Разница небольшая, потому что мы сделали только один шаг с маленькой скоростью обучения.

## 10. Обучение до сходимости

После нескольких эпох векторы стабилизируются. Приведём примерные итоговые значения после 5 эпох (округлённо до 2 знаков).

**Входные векторы $U$ (итоговые эмбеддинги):**

| Слово | $u_w$ |
|-------|-------|
| кошка | $(0.43, -0.29)$ |
| сидит | $(0.36, 0.52)$ |
| на | $(-0.17, 0.61)$ |
| окне | $(0.39, -0.18)$ |
| собака | $(0.40, -0.26)$ |
| крыльце | $(-0.32, 0.13)$ |
| спит | $(0.54, 0.47)$ |
| диване | $(-0.22, -0.40)$ |

**Выходные векторы $V$:**

| Слово | $v_w$ |
|-------|-------|
| кошка | $(0.17, 0.34)$ |
| сидит | $(-0.14, 0.44)$ |
| на | $(0.51, -0.09)$ |
| окне | $(0.31, 0.22)$ |
| собака | $(-0.29, -0.19)$ |
| крыльце | $(0.39, 0.47)$ |
| спит | $(-0.09, 0.57)$ |
| диване | $(0.21, -0.39)$ |

**Интерпретация:**

- Векторы «кошка» и «собака» близки: $(0.43, -0.29)$ и $(0.40, -0.26)$. Оба слова встречаются с «сидит».
- Векторы «спит» и «диване»: $(0.54, 0.47)$ и $(-0.22, -0.40)$ — далеки, хотя встречаются вместе. Это артефакт крошечного корпуса.
- Векторы «окне» и «крыльце»: $(0.39, -0.18)$ и $(-0.32, 0.13)$ — не очень близки, но оба связаны с «на».

## 11. Итоговая матрица вероятностей $P(w_t \mid C)$

По обученным векторам можно вычислить вероятность каждого целевого слова для каждого контекста. Приведём матрицу для нескольких контекстов (округлённо до 3 знаков).

**Для контекста $\{$кошка, на$\}$:**

$$
h = \frac{1}{2} \left[ (0.43, -0.29) + (-0.17, 0.61) \right] = \frac{1}{2} (0.26, 0.32) = (0.13, 0.16).
$$

Скалярные произведения с выходными векторами:

| Целевое слово | $v_w^\top h$ | $\exp(v_w^\top h)$ | $P(w \mid C)$ |
|---------------|--------------|---------------------|----------------|
| кошка | $0.17 \cdot 0.13 + 0.34 \cdot 0.16 = 0.022 + 0.054 = 0.076$ | $1.079$ | $0.124$ |
| сидит | $-0.14 \cdot 0.13 + 0.44 \cdot 0.16 = -0.018 + 0.070 = 0.052$ | $1.053$ | $0.121$ |
| на | $0.51 \cdot 0.13 + (-0.09) \cdot 0.16 = 0.066 - 0.014 = 0.052$ | $1.053$ | $0.121$ |
| окне | $0.31 \cdot 0.13 + 0.22 \cdot 0.16 = 0.040 + 0.035 = 0.075$ | $1.078$ | $0.124$ |
| собака | $-0.29 \cdot 0.13 + (-0.19) \cdot 0.16 = -0.038 - 0.030 = -0.068$ | $0.934$ | $0.107$ |
| крыльце | $0.39 \cdot 0.13 + 0.47 \cdot 0.16 = 0.051 + 0.075 = 0.126$ | $1.134$ | $0.130$ |
| спит | $-0.09 \cdot 0.13 + 0.57 \cdot 0.16 = -0.012 + 0.091 = 0.079$ | $1.082$ | $0.124$ |
| диване | $0.21 \cdot 0.13 + (-0.39) \cdot 0.16 = 0.027 - 0.062 = -0.035$ | $0.966$ | $0.111$ |

Сумма экспонент:

$$
Z = 1.079 + 1.053 + 1.053 + 1.078 + 0.934 + 1.134 + 1.082 + 0.966 = 8.379.
$$

Вероятности (нормированные):

| Целевое слово | $P(w \mid C)$ |
|---------------|----------------|
| кошка | $0.129$ |
| сидит | $0.126$ |
| на | $0.126$ |
| окне | $0.129$ |
| собака | $0.111$ |
| крыльце | $0.135$ |
| спит | $0.129$ |
| диване | $0.115$ |

**Наблюдение:** вероятности всё ещё близки, потому что корпус крошечный. В реальных задачах правильное целевое слово получает вероятность 0.3–0.5, остальные — 0.001–0.01.

## 12. Сравнение с Skip-gram на том же корпусе

Проведём сравнение CBOW и Skip-gram на одном и том же корпусе.

**Число обучающих пар за эпоху:**

| Архитектура | Число пар |
|-------------|-----------|
| CBOW | 12 |
| Skip-gram | 23 |

**Число обновлений для слова «сидит» за эпоху:**

- **CBOW:** «сидит» встречается как целевое слово в позициях 2 и 6 (два раза) и как контекстное слово в позициях 3, 5, 7 (три раза). Итого 5 обновлений.
- **Skip-gram:** «сидит» встречается как целевое слово в позициях 2 и 6 (два раза, каждое даёт 2 пары = 4 пары) и как контекстное слово в 5 парах. Итого 9 обновлений.

**Ключевое отличие:** Skip-gram даёт больше обновлений для каждого слова, особенно для редких. CBOW быстрее, но менее чувствителен к редким словам.

**Временная сложность:**

- CBOW: $O(T \cdot d \cdot (2m + K))$ — усреднение плюс $K+1$ скалярных произведений.
- Skip-gram: $O(T \cdot 2m \cdot d \cdot (1 + K))$ — в $2m$ раз больше пар.

**Качество:**

- CBOW лучше для частых слов (усреднение сглаживает шум).
- Skip-gram лучше для редких слов (больше обновлений).
- CBOW быстрее.
- Skip-gram даёт более «семантические» векторы, CBOW — более «синтаксические».

## 13. Заключение

В этом численном примере мы шаг за шагом вычислили CBOW для учебного корпуса. Основные выводы:

1. **CBOW генерирует одну пару на позицию** — контекст → целевое слово. Это делает его быстрее Skip-gram.

2. **Усреднение контекстных векторов** — ключевая операция CBOW. Все контекстные слова получают одинаковое обновление, потому что градиент по $h$ распределяется равномерно.

3. **Целевое слово обновляет только выходной вектор** (в CBOW), а не входной. В Skip-gram целевое слово обновляет входной вектор.

4. **Negative sampling позволяет избежать softmax по всему словарю.** Мы вычисляем только $K+1$ скалярных произведений вместо $N$.

5. **Градиенты имеют простую форму:** $(1 - \sigma(x)) h$ для положительной пары, $-\sigma(x') h$ для отрицательной, и их комбинация для $h$.

6. **После обучения семантически близкие слова имеют близкие векторы.** «Кошка» и «собака» оказываются рядом.

**Ключевые формулы:**

Усреднённый контекст:

$$
h = \frac{1}{2m} \sum_{c \in C} u_c.
$$

Вероятность целевого слова:

$$
P(w_t \mid C) = \frac{\exp(v_{w_t}^\top h)}{\sum_{w \in V} \exp(v_w^\top h)}.
$$

Функция потерь с negative sampling:

$$
\mathcal{L} = \log \sigma(v_{w_t}^\top h) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top h).
$$

Градиенты:

$$
\frac{\partial \mathcal{L}}{\partial v_{w_t}} = (1 - \sigma(x)) h,
$$

$$
\frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}}}} = -\sigma(x') h,
$$

$$
\frac{\partial \mathcal{L}}{\partial u_{c_i}} = \frac{1}{2m} \left[ (1 - \sigma(x)) v_{w_t} - \sigma(x') v_{w_{\text{neg}}} \right].
$$

Эти формулы — основа CBOW. Их понимание позволяет эффективно обучать эмбеддинги и применять их в реальных задачах.

# Skip-gram: подробная теория

## Введение: почему Skip-gram заслуживает отдельного разбора

В предыдущей части мы подробно разобрали CBOW — архитектуру Word2Vec, которая предсказывает целевое слово по контексту. CBOW быстра, устойчива к шуму и хорошо работает для частых слов. Но у неё есть принципиальное ограничение: усреднение контекстных векторов «размывает» информацию о редких словах. Если редкое слово встречается в корпусе всего несколько раз, его вклад в усреднённый вектор $h$ теряется среди вкладов частых слов. В результате вектор редкого слова обучается плохо.

Skip-gram решает эту проблему. Вместо того чтобы усреднять контекст и предсказывать одно слово, Skip-gram использует **одно целевое слово** для предсказания **каждого контекстного слова по отдельности**. Это означает, что каждое вхождение слова в корпус даёт не одну, а $2m$ обучающих пар. Для редких слов это критически важно: даже если слово встречается всего 10 раз, Skip-gram генерирует $10 \times 2m$ обновлений, что позволяет выучить осмысленный вектор.

Именно поэтому в оригинальной статье Word2Vec Skip-gram рекомендуется для маленьких корпусов и задач, где важны редкие слова. На больших корпусах Skip-gram тоже работает отлично, но требует больше времени на обучение.

В этой лекции мы подробно разберём Skip-gram: архитектуру, функцию правдоподобия, проблему softmax, negative sampling с полным выводом градиентов, иерархический softmax, субсэмплирование и практические рекомендации. Мы будем следовать той же структуре, что и в лекции по CBOW, но с акцентом на особенности Skip-gram.

---

## 1. Формальное определение задачи

### 1.1 Постановка задачи и обозначения

Пусть дан корпус — последовательность слов:

$$
w_1, w_2, \ldots, w_T,
$$

где $T$ — длина корпуса. Словарь:

$$
V = \{w_1, w_2, \ldots, w_N\},
$$

где $N = |V|$ — размер словаря. Зафиксируем размер окна $m$. Для каждой позиции $t$ целевое слово — это $w_t$, а контекстные слова — это слова в окне вокруг $w_t$:

$$
w_{t-m}, \ldots, w_{t-1}, w_{t+1}, \ldots, w_{t+m}.
$$

Обозначим контекст через $C_t$:

$$
C_t = \{w_{t-m}, \ldots, w_{t-1}, w_{t+1}, \ldots, w_{t+m}\}.
$$

Число контекстных слов равно $2m$.

Рассмотрим конкретный пример. Пусть корпус — это объединённая последовательность наших трёх документов:

$$
\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{сидит}, \text{на}, \text{крыльце}, \text{кошка}, \text{спит}, \text{на}, \text{диване}.
$$

Длина последовательности $T = 12$. Пусть окно $m = 1$. Тогда для каждой позиции $t$ контекст — это слова на позициях $t-1$ и $t+1$.

| Позиция $t$ | Целевое $w_t$ | Контекст $C_t$ |
|-------------|---------------|----------------|
| 1 | кошка | $\{$сидит$\}$ |
| 2 | сидит | $\{$кошка, на$\}$ |
| 3 | на | $\{$сидит, окне$\}$ |
| 4 | окне | $\{$на, собака$\}$ |
| 5 | собака | $\{$окне, сидит$\}$ |
| 6 | сидит | $\{$собака, на$\}$ |
| 7 | на | $\{$сидит, крыльце$\}$ |
| 8 | крыльце | $\{$на, кошка$\}$ |
| 9 | кошка | $\{$крыльце, спит$\}$ |
| 10 | спит | $\{$кошка, на$\}$ |
| 11 | на | $\{$спит, диване$\}$ |
| 12 | диване | $\{$на$\}$ |

**Задача Skip-gram:** для каждой позиции $t$ и каждого контекстного слова $w_{t+j} \in C_t$ предсказать $w_{t+j}$ по целевому слову $w_t$. Иными словами, мы максимизируем вероятность $P(w_{t+j} \mid w_t)$ для всех $j \in \{-m, \ldots, -1, 1, \ldots, m\}$.

**Ключевое отличие от CBOW:** в CBOW мы для каждой позиции генерируем **одну** обучающую пару (контекст → целевое слово). В Skip-gram мы генерируем **$2m$** обучающих пар (целевое слово → каждое контекстное слово). Это делает Skip-gram в $2m$ раз медленнее на одну эпоху, но даёт больше обновлений для каждого слова.

### 1.2 Параметры модели

Как и в CBOW, Skip-gram имеет два набора параметров:

1. **Входные векторы (input vectors):** матрица $U \in \mathbb{R}^{N \times d}$. Строка $i$ матрицы $U$ — это вектор $u_{w_i} \in \mathbb{R}^d$ слова $w_i$. Эти векторы используются для **целевого** слова.

2. **Выходные векторы (output vectors):** матрица $V \in \mathbb{R}^{N \times d}$. Строка $i$ матрицы $V$ — это вектор $v_{w_i} \in \mathbb{R}^d$ слова $w_i$. Эти векторы используются для **контекстных** слов.

**Тонкий момент:** в CBOW входные векторы использовались для контекстных слов, а выходные — для целевого. В Skip-gram наоборот: входные — для целевого, выходные — для контекстных. Это отражает разницу в направлении предсказания.

Почему нужны два набора векторов? Та же причина, что и в CBOW: роли слова в паре (целевое/контекстное) асимметричны. Если бы мы использовали один вектор $w_w$, то вероятность $P(w_O \mid w_I)$ была бы симметричной относительно перестановки целевого и контекстного слов. Но в реальности вероятность встретить «кошку» рядом с «сидит» не равна вероятности встретить «сидит» рядом с «кошкой». Разделение на входные и выходные векторы позволяет модели уловить эту асимметрию.

После обучения обычно используют либо $u_w$, либо $v_w$, либо их сумму $u_w + v_w$ как итоговый эмбеддинг. На практике $u_w$ и $v_w$ дают близкие результаты, но $u_w$ используется чаще.

### 1.3 Прямой проход: шаг за шагом

Рассмотрим, как Skip-gram вычисляет вероятность контекстного слова по целевому. Мы разберём каждый шаг подробно.

#### Шаг 1: Получение вектора целевого слова

Для целевого слова $w_I = w_t$ берём его входной вектор $u_{w_I} \in \mathbb{R}^d$.

**Пример:** целевое слово «сидит». Пусть

$$
u_{\text{сидит}} = (0.3, 0.4).
$$

Здесь $d = 2$ для наглядности. В реальных задачах $d = 100$–$300$.

#### Шаг 2: Вычисление скалярного произведения с каждым контекстным словом

Для каждого контекстного слова $w_O \in C_t$ вычисляем скалярное произведение:

$$
s(w_I, w_O) = v_{w_O}^\top u_{w_I} = \sum_{k=1}^{d} v_{w_O, k} \cdot u_{w_I, k}.
$$

Разберём формулу. $v_{w_O} \in \mathbb{R}^d$ — выходной вектор контекстного слова. $u_{w_I} \in \mathbb{R}^d$ — входной вектор целевого слова. Скалярное произведение — это сумма покомпонентных произведений:

$$
v_{w_O}^\top u_{w_I} = v_{w_O, 1} \cdot u_{w_I, 1} + v_{w_O, 2} \cdot u_{w_I, 2} + \ldots + v_{w_O, d} \cdot u_{w_I, d}.
$$

Результат — скаляр, мера совместимости контекстного слова с целевым.

**Интуиция:** если выходной вектор контекстного слова близок к входному вектору целевого слова (указывает в том же направлении), скалярное произведение велико и положительно. Это означает, что контекстное слово хорошо «подходит» к данному целевому слову. Если векторы ортогональны, произведение равно нулю. Если противоположны, отрицательно.

**Пример:** для контекстного слова «кошка» с $v_{\text{кошка}} = (0.1, 0.3)$:

$$
s(\text{сидит}, \text{кошка}) = 0.1 \cdot 0.3 + 0.3 \cdot 0.4 = 0.03 + 0.12 = 0.15.
$$

Для контекстного слова «на» с $v_{\text{на}} = (0.5, -0.1)$:

$$
s(\text{сидит}, \text{на}) = 0.5 \cdot 0.3 + (-0.1) \cdot 0.4 = 0.15 - 0.04 = 0.11.
$$

#### Шаг 3: Softmax

Вероятность контекстного слова $w_O$ при условии целевого $w_I$:

$$
P(w_O \mid w_I) = \frac{\exp(s(w_I, w_O))}{\sum_{w \in V} \exp(s(w_I, w))} = \frac{\exp(v_{w_O}^\top u_{w_I})}{\sum_{w \in V} \exp(v_w^\top u_{w_I})}.
$$

Разберём формулу по частям.

**Числитель:** $\exp(s(w_I, w_O)) = \exp(v_{w_O}^\top u_{w_I})$. Экспонента делает значение положительным и усиливает различия.

**Знаменатель:** $\sum_{w \in V} \exp(s(w_I, w)) = \sum_{w \in V} \exp(v_w^\top u_{w_I})$. Сумма экспонент по всему словарю. Она нормирует вероятность так, чтобы сумма по всем возможным контекстным словам была равна единице.

**Свойства softmax:**

- $P(w_O \mid w_I) > 0$ для всех $w_O$;
- $\sum_{w \in V} P(w \mid w_I) = 1$;
- Если $s(w_I, w_O)$ велико по сравнению с другими $s(w_I, w)$, то $P(w_O \mid w_I) \approx 1$.

**Пример:** пусть словарь $\{$кошка, сидит, на, окне$\}$, $u_{\text{сидит}} = (0.3, 0.4)$.

Выходные векторы:

$$
v_{\text{кошка}} = (0.1, 0.3), \quad v_{\text{сидит}} = (-0.2, 0.4), \quad v_{\text{на}} = (0.5, -0.1), \quad v_{\text{окне}} = (0.3, 0.2).
$$

Скалярные произведения:

$$
s(\text{сидит}, \text{кошка}) = 0.1 \cdot 0.3 + 0.3 \cdot 0.4 = 0.03 + 0.12 = 0.15,
$$

$$
s(\text{сидит}, \text{сидит}) = -0.2 \cdot 0.3 + 0.4 \cdot 0.4 = -0.06 + 0.16 = 0.10,
$$

$$
s(\text{сидит}, \text{на}) = 0.5 \cdot 0.3 + (-0.1) \cdot 0.4 = 0.15 - 0.04 = 0.11,
$$

$$
s(\text{сидит}, \text{окне}) = 0.3 \cdot 0.3 + 0.2 \cdot 0.4 = 0.09 + 0.08 = 0.17.
$$

Экспоненты:

$$
\exp(0.15) \approx 1.162, \quad \exp(0.10) \approx 1.105, \quad \exp(0.11) \approx 1.116, \quad \exp(0.17) \approx 1.185.
$$

Сумма:

$$
Z = 1.162 + 1.105 + 1.116 + 1.185 = 4.568.
$$

Вероятности:

$$
P(\text{кошка} \mid \text{сидит}) = 1.162 / 4.568 \approx 0.254,
$$

$$
P(\text{сидит} \mid \text{сидит}) = 1.105 / 4.568 \approx 0.242,
$$

$$
P(\text{на} \mid \text{сидит}) = 1.116 / 4.568 \approx 0.244,
$$

$$
P(\text{окне} \mid \text{сидит}) = 1.185 / 4.568 \approx 0.259.
$$

**Наблюдение:** все вероятности близки к $1/4 = 0.25$, потому что векторы ещё не обучены. После обучения вероятности станут более контрастными: контекстные слова, которые действительно встречаются рядом с «сидит», будут иметь высокую вероятность, остальные — низкие.

---

## 2. Функция правдоподобия и её вывод

### 2.1 Правдоподобие для одной пары

Для одной пары (целевое слово $w_I$, контекстное слово $w_O$) вероятность:

$$
P(w_O \mid w_I) = \frac{\exp(v_{w_O}^\top u_{w_I})}{\sum_{w \in V} \exp(v_w^\top u_{w_I})}.
$$

**Логарифм правдоподобия** для одной пары:

$$
\log P(w_O \mid w_I) = v_{w_O}^\top u_{w_I} - \log \sum_{w \in V} \exp(v_w^\top u_{w_I}).
$$

Разберём эту формулу. Мы взяли логарифм от дроби:

$$
\log \frac{\exp(v_{w_O}^\top u_{w_I})}{\sum_{w} \exp(v_w^\top u_{w_I})} = \log \exp(v_{w_O}^\top u_{w_I}) - \log \sum_{w} \exp(v_w^\top u_{w_I}).
$$

Поскольку $\log \exp(x) = x$, первое слагаемое равно $v_{w_O}^\top u_{w_I}$. Второе слагаемое — логарифм суммы экспонент.

**Интерпретация:**

- Первое слагаемое $v_{w_O}^\top u_{w_I}$ — скалярное произведение для правильной пары. Мы хотим его максимизировать.
- Второе слагаемое $\log \sum_{w} \exp(v_w^\top u_{w_I})$ — логарифм суммы экспонент. Мы хотим его минимизировать.

### 2.2 Правдоподобие для всего корпуса

Для всего корпуса правдоподобие — это произведение вероятностей всех наблюдаемых пар:

$$
\mathcal{L}(U, V) = \prod_{t=1}^{T} \prod_{-m \le j \le m, j \ne 0} P(w_{t+j} \mid w_t).
$$

Мы предполагаем, что все пары независимы при фиксированных параметрах. Это упрощающее предположение, но оно позволяет записать правдоподобие в такой простой форме.

**Логарифм правдоподобия:**

$$
\ell(U, V) = \log \mathcal{L} = \sum_{t=1}^{T} \sum_{-m \le j \le m, j \ne 0} \log P(w_{t+j} \mid w_t).
$$

Подставляем выражение для $P$:

$$
\ell(U, V) = \sum_{t=1}^{T} \sum_{-m \le j \le m, j \ne 0} \left[ v_{w_{t+j}}^\top u_{w_t} - \log \sum_{w \in V} \exp(v_w^\top u_{w_t}) \right].
$$

**Цель:** найти $U$ и $V$, максимизирующие $\ell$.

### 2.3 Свойства функции правдоподобия

**Невыпуклость.** Функция $\ell$ невыпукла по $U$ и $V$ из-за наличия softmax. Это означает, что глобальный максимум не гарантирован. На практике используется стохастический градиентный подъём (SGD), который сходится к локальному максимуму.

**Симметрия.** Если переставить слова в словаре, значение $\ell$ не изменится (при соответствующей перестановке строк $U$ и $V$). Это свойство идентифицируемости: решение не единственно.

**Масштаб.** Если умножить все векторы на константу $c > 0$, значение $\ell$ изменится. Это означает, что нормы векторов не определены однозначно. После обучения векторы часто нормализуют по L2.

---

## 3. Проблема вычислительной сложности softmax

### 3.1 Вычислительная стоимость

Знаменатель softmax:

$$
Z(w_I) = \sum_{w \in V} \exp(v_w^\top u_{w_I})
$$

требует суммирования по **всему словарю** $N$. Для каждого слова $w$ нужно:

1. Вычислить скалярное произведение $v_w^\top u_{w_I}$: $d$ умножений и $d-1$ сложений.
2. Вычислить экспоненту $\exp(v_w^\top u_{w_I})$.

Итого $O(N \cdot d)$ операций на одну пару.

**Оценка:** если $N = 10^6$, $d = 300$, $T = 10^9$ (корпус из миллиарда слов), а окно $m = 5$, то общее число пар:

$$
T \times 2m = 10^9 \times 10 = 10^{10}.
$$

Для каждой пары $10^6 \times 300 = 3 \times 10^8$ операций. Итого:

$$
10^{10} \times 3 \times 10^8 = 3 \times 10^{18}
$$

операций. Это на порядок больше, чем в CBOW, потому что Skip-gram генерирует в $2m$ раз больше пар.

### 3.2 Решения

Существуют три основных подхода:

1. **Negative sampling:** заменить softmax на бинарную классификацию с $K$ отрицательными примерами. Сложность $O(K \cdot d)$ на пару.
2. **Иерархический softmax:** заменить плоский softmax на дерево Хаффмана. Сложность $O(\log N \cdot d)$ на пару.
3. **Субсэмплирование:** уменьшить число пар, отбрасывая частые слова.

Рассмотрим каждый подход подробно.

---

## 4. Skip-gram с negative sampling

### 4.1 Идея negative sampling

Negative sampling заменяет задачу предсказания вероятности $P(w_O \mid w_I)$ через softmax по всему словарю на задачу **бинарной классификации**:

- **Положительный пример:** реальная пара $(w_I, w_O)$ из корпуса. Метка: $D = 1$.
- **Отрицательные примеры:** пары $(w_I, w_{\text{neg}_1}), \ldots, (w_I, w_{\text{neg}_K})$, где $w_{\text{neg}_k}$ — случайные слова. Метка: $D = 0$.

Мы обучаем модель различать реальные пары от случайных.

**Ключевое преимущество:** вместо вычисления $N$ скалярных произведений (по одному на каждое слово словаря) мы вычисляем только $K+1$ скалярных произведений. Поскольку $K \ll N$, это даёт огромное ускорение.

### 4.2 Вероятностная модель

Вероятность того, что пара $(w_I, w_O)$ является реальной:

$$
P(D = 1 \mid w_I, w_O) = \sigma(v_{w_O}^\top u_{w_I}),
$$

где $\sigma(x) = \frac{1}{1 + e^{-x}}$ — сигмоида.

Разберём эту формулу. Сигмоида преобразует скалярное произведение (которое может быть любым числом от $-\infty$ до $+\infty$) в вероятность от 0 до 1. Если скалярное произведение велико и положительно, сигмоида близка к 1. Если велико и отрицательно, близка к 0.

Вероятность того, что пара $(w_I, w_{\text{neg}})$ является случайной:

$$
P(D = 0 \mid w_I, w_{\text{neg}}) = 1 - \sigma(v_{w_{\text{neg}}}^\top u_{w_I}) = \sigma(-v_{w_{\text{neg}}}^\top u_{w_I}).
$$

**Свойство сигмоиды:** $\sigma(x) + \sigma(-x) = 1$. Это легко проверить:

$$
\sigma(x) + \sigma(-x) = \frac{1}{1+e^{-x}} + \frac{1}{1+e^{x}} = \frac{1+e^x + 1+e^{-x}}{(1+e^{-x})(1+e^x)} = \frac{2 + e^x + e^{-x}}{2 + e^x + e^{-x}} = 1.
$$

### 4.3 Функция потерь

Для одной реальной пары $(w_I, w_O)$ с $K$ отрицательными примерами:

$$
\mathcal{L}(w_I, w_O) = \log \sigma(v_{w_O}^\top u_{w_I}) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top u_{w_I}).
$$

Разберём по частям.

**Первое слагаемое:** $\log \sigma(v_{w_O}^\top u_{w_I})$. Это логарифм вероятности, что реальная пара правильно классифицирована. Мы хотим его максимизировать. Поскольку $\log$ — монотонно возрастающая функция, максимизация $\log \sigma(x)$ эквивалентна максимизации $\sigma(x)$, что эквивалентно максимизации $x = v_{w_O}^\top u_{w_I}$.

**Второе слагаемое:** $\sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top u_{w_I})$. Это сумма логарифмов вероятностей, что каждое из $K$ случайных слов правильно классифицировано как случайное. Максимизация $\log \sigma(-x')$ эквивалентна максимизации $\sigma(-x')$, что эквивалентно минимизации $x' = v_{w_{\text{neg}}}^\top u_{w_I}$.

**Для всего корпуса:**

$$
\mathcal{L}(U, V) = \sum_{(w_I, w_O) \in \mathcal{D}} \left[ \log \sigma(v_{w_O}^\top u_{w_I}) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top u_{w_I}) \right],
$$

где $\mathcal{D}$ — множество всех пар (целевое, контекстное) из корпуса.

**Цель:** максимизировать $\mathcal{L}$ по $U$ и $V$.

### 4.4 Интуиция за функцией потерь

Рассмотрим, что происходит при максимизации каждого слагаемого.

**Первое слагаемое:** $\log \sigma(x)$, где $x = v_{w_O}^\top u_{w_I}$. Функция $\sigma(x)$ монотонно возрастает от 0 до 1. Логарифм $\log \sigma(x)$ тоже монотонно возрастает. Максимум достигается при $x \to \infty$. Это означает, что модель хочет сделать скалярное произведение $v_{w_O}^\top u_{w_I}$ как можно большим для реальных пар. Геометрически это означает, что выходной вектор контекстного слова должен быть близок к входному вектору целевого слова.

**Второе слагаемое:** $\log \sigma(-x')$, где $x' = v_{w_{\text{neg}}}^\top u_{w_I}$. Функция $\sigma(-x')$ монотонно убывает по $x'$. Логарифм тоже монотонно убывает. Максимум достигается при $x' \to -\infty$. Это означает, что модель хочет сделать скалярное произведение $v_{w_{\text{neg}}}^\top u_{w_I}$ как можно меньшим для случайных пар. Геометрически это означает, что выходные векторы случайных слов должны быть далеки от входного вектора целевого слова.

Таким образом, обучение с negative sampling одновременно притягивает выходной вектор правильного контекстного слова к входному вектору целевого слова и отталкивает выходные векторы случайных слов от входного вектора целевого слова.

### 4.5 Выбор отрицательных примеров

Отрицательные примеры выбираются из **шумового распределения**:

$$
P_n(w) \propto \text{count}(w)^{3/4},
$$

где $\text{count}(w)$ — частота слова $w$ в корпусе.

**Нормированное распределение:**

$$
P_n(w) = \frac{\text{count}(w)^{3/4}}{\sum_{w' \in V} \text{count}(w')^{3/4}}.
$$

Почему именно степень 3/4? Мы уже обсуждали это в лекции по CBOW. Кратко:

- При степени 1 (униграммное распределение) частые слова выбираются слишком часто.
- При степени 0 (равномерное распределение) редкие слова выбираются слишком часто.
- Степень 3/4 — эмпирический компромисс, который хорошо работает на практике.

### 4.6 Количество отрицательных примеров $K$

**Рекомендации:**

- Маленькие корпуса: $K = 5$–$20$.
- Большие корпуса: $K = 2$–$5$.

**Тонкий момент:** больше $K$ — точнее градиент, но медленнее обучение. Меньше $K$ — быстрее, но шумнее.

---

## 5. Градиенты для Skip-gram с negative sampling

### 5.1 Обозначения

Для одной пары $(w_I, w_O)$ с $K$ отрицательными примерами:

- $u = u_{w_I} \in \mathbb{R}^d$ — входной вектор целевого слова;
- $v = v_{w_O} \in \mathbb{R}^d$ — выходной вектор реального контекстного слова;
- $v'_k = v_{w_{\text{neg}_k}} \in \mathbb{R}^d$ — выходной вектор $k$-го отрицательного слова;
- $x = v^\top u$ — скалярное произведение для положительной пары;
- $x'_k = (v'_k)^\top u$ — скалярное произведение для $k$-й отрицательной пары.

### 5.2 Функция потерь

$$
\mathcal{L} = \log \sigma(x) + \sum_{k=1}^{K} \log \sigma(-x'_k).
$$

### 5.3 Производные сигмоиды

Вспомним:

$$
\sigma(x) = \frac{1}{1 + e^{-x}}.
$$

Производная:

$$
\sigma'(x) = \frac{d}{dx} \sigma(x) = \frac{e^{-x}}{(1 + e^{-x})^2} = \sigma(x)(1 - \sigma(x)).
$$

### 5.4 Производная логарифма сигмоиды

$$
\frac{\partial}{\partial x} \log \sigma(x) = \frac{\sigma'(x)}{\sigma(x)} = 1 - \sigma(x).
$$

$$
\frac{\partial}{\partial x'} \log \sigma(-x') = \frac{-\sigma'(-x')}{\sigma(-x')} = -(1 - \sigma(-x')) = -\sigma(x').
$$

### 5.5 Градиент по выходному вектору $v$

$$
\frac{\partial \mathcal{L}}{\partial v} = \frac{\partial \mathcal{L}}{\partial x} \cdot \frac{\partial x}{\partial v} = (1 - \sigma(x)) \cdot u.
$$

### 5.6 Градиент по выходному вектору $v'_k$

$$
\frac{\partial \mathcal{L}}{\partial v'_k} = \frac{\partial \mathcal{L}}{\partial x'_k} \cdot \frac{\partial x'_k}{\partial v'_k} = -\sigma(x'_k) \cdot u.
$$

### 5.7 Градиент по входному вектору $u$

$$
\frac{\partial \mathcal{L}}{\partial u} = \frac{\partial \mathcal{L}}{\partial x} \cdot \frac{\partial x}{\partial u} + \sum_{k=1}^{K} \frac{\partial \mathcal{L}}{\partial x'_k} \cdot \frac{\partial x'_k}{\partial u}.
$$

Вычислим:

$$
\frac{\partial x}{\partial u} = \frac{\partial (v^\top u)}{\partial u} = v,
$$

$$
\frac{\partial x'_k}{\partial u} = \frac{\partial ((v'_k)^\top u)}{\partial u} = v'_k.
$$

Тогда:

$$
\frac{\partial \mathcal{L}}{\partial u} = (1 - \sigma(x)) v - \sum_{k=1}^{K} \sigma(x'_k) v'_k.
$$

### 5.8 Обновление параметров

Используя градиентный подъём:

$$
v \leftarrow v + \eta (1 - \sigma(x)) u,
$$

$$
v'_k \leftarrow v'_k - \eta \sigma(x'_k) u, \quad k = 1, \ldots, K,
$$

$$
u \leftarrow u + \eta \left[ (1 - \sigma(x)) v - \sum_{k=1}^{K} \sigma(x'_k) v'_k \right].
$$

### 5.9 Интерпретация обновлений

**Выходной вектор $v$ контекстного слова:** обновляется в направлении $u$ с коэффициентом $(1 - \sigma(x))$. Если модель уже уверена, что пара реальна ($\sigma(x) \approx 1$), обновление слабое. Если модель ошибается ($\sigma(x) \approx 0$), обновление сильное.

**Выходные векторы $v'_k$ отрицательных слов:** обновляются в направлении $-u$ с коэффициентом $\sigma(x'_k)$. Если модель ошибочно считает отрицательную пару реальной ($\sigma(x'_k) \approx 1$), обновление сильное.

**Входной вектор $u$ целевого слова:** обновляется в направлении $v$ (притягивается к реальному контексту) и в направлении $-v'_k$ (отталкивается от отрицательных). Это ключевое отличие от CBOW: в Skip-gram целевое слово получает обновление входного вектора для **каждой** пары. Если слово встречается в корпусе часто, оно получает много обновлений.

### 5.10 Полный алгоритм Skip-gram с negative sampling

1. **Инициализация:** случайные малые значения для $U$ и $V$.

2. **Для каждой эпохи:**
   - Для каждой позиции $t$ в корпусе:
     - Целевое слово: $w_I = w_t$.
     - Для каждого $j \in \{-m, \ldots, -1, 1, \ldots, m\}$:
       - Контекстное слово: $w_O = w_{t+j}$.
       - Выбрать $K$ отрицательных примеров $w_{\text{neg}_1}, \ldots, w_{\text{neg}_K}$ из $P_n(w)$.
       - Вычислить $x = v_{w_O}^\top u_{w_I}$ и $x'_k = v_{w_{\text{neg}_k}}^\top u_{w_I}$.
       - Обновить векторы по формулам выше.

3. **Повторять** до сходимости.

**Тонкий момент:** в Skip-gram на одну позицию приходится $2m$ обучающих пар. Это делает Skip-gram в $2m$ раз медленнее CBOW на одну эпоху, но даёт больше обновлений для каждого слова.

---

## 6. Skip-gram с иерархическим softmax

### 6.1 Идея

Иерархический softmax использует бинарное дерево (дерево Хаффмана), в листьях которого находятся слова. Вероятность контекстного слова вычисляется как произведение вероятностей на пути от корня до листа.

**Преимущество:** сложность $O(\log N)$ вместо $O(N)$.

### 6.2 Построение дерева Хаффмана

1. Каждое слово — лист дерева. Вес листа — частота слова в корпусе.
2. На каждом шаге выбираются два узла с наименьшими весами и объединяются в новый узел с суммой весов.
3. Процесс повторяется, пока не останется один корневой узел.

**Результат:** частые слова ближе к корню (короткие пути), редкие — дальше (длинные пути).

### 6.3 Формула вероятности

Пусть путь от корня до слова $w_O$ состоит из узлов $n_1, \ldots, n_L$. На каждом узле $n_i$ есть бинарный выбор $d_i \in \{0, 1\}$ (0 — налево, 1 — направо).

Вероятность контекстного слова $w_O$ при условии целевого $w_I$:

$$
P(w_O \mid w_I) = \prod_{i=1}^{L} \sigma\left( (-1)^{d_i} v_{n_i}^\top u_{w_I} \right).
$$

**Разберём:**

- Если $d_i = 0$ (налево): $(-1)^{d_i} = 1$, вероятность $\sigma(v_{n_i}^\top u_{w_I})$.
- Если $d_i = 1$ (направо): $(-1)^{d_i} = -1$, вероятность $\sigma(-v_{n_i}^\top u_{w_I})$.

**Интуиция:** на каждом узле мы делаем бинарный выбор. Вероятность слова — это произведение вероятностей всех выборов на пути.

### 6.4 Функция потерь

Для одной пары $(w_I, w_O)$:

$$
\log P(w_O \mid w_I) = \sum_{i=1}^{L_O} \log \sigma\left( (-1)^{d_i^O} v_{n_i^O}^\top u_{w_I} \right),
$$

где $L_O$ — длина пути до слова $w_O$.

Для всего корпуса:

$$
\ell = \sum_{(w_I, w_O) \in \mathcal{D}} \sum_{i=1}^{L_O} \log \sigma\left( (-1)^{d_i^O} v_{n_i^O}^\top u_{w_I} \right).
$$

### 6.5 Градиенты

Для каждого узла $n_i$ на пути:

$$
z_i = (-1)^{d_i} v_{n_i}^\top u_{w_I}.
$$

Градиент по $v_{n_i}$:

$$
\frac{\partial \ell}{\partial v_{n_i}} = (-1)^{d_i} (1 - \sigma(z_i)) u_{w_I}.
$$

Градиент по $u_{w_I}$:

$$
\frac{\partial \ell}{\partial u_{w_I}} = \sum_{i=1}^{L_O} (-1)^{d_i} (1 - \sigma(z_i)) v_{n_i}.
$$

### 6.6 Сложность

- Плоский softmax: $O(N)$.
- Иерархический softmax: $O(\log N)$.

**Пример:** $N = 10^6$, $\log_2 N \approx 20$. Ускорение в 50 000 раз.

---

## 7. Субсэмплирование частых слов

### 7.1 Проблема

Частые слова (артикли, предлоги, союзы) встречаются в корпусе миллионы раз. Они дают много обучающих пар, но не несут семантической информации. В Skip-gram это особенно критично, потому что каждое вхождение частого слова генерирует $2m$ пар.

### 7.2 Формула

Вероятность **удалить** слово $w$:

$$
P_{\text{discard}}(w) = 1 - \sqrt{\frac{t}{f(w)}},
$$

где $f(w)$ — частота слова в корпусе, $t$ — порог (обычно $t = 10^{-5}$).

### 7.3 Пример

$t = 10^{-5}$.

- Слово «the»: $f = 0.05$. $P_{\text{discard}} = 1 - \sqrt{10^{-5}/0.05} \approx 0.986$. Удаляется с вероятностью 98.6%.
- Слово «кошка»: $f = 10^{-5}$. $P_{\text{discard}} = 0$. Сохраняется.
- Слово «диван»: $f = 10^{-6}$. $P_{\text{discard}} < 0$. Сохраняется.

### 7.4 Влияние

- Ускоряет обучение в 2–10 раз.
- Улучшает качество векторов для редких слов.
- Удаляет шум от частых слов.

---

## 8. Сравнение Skip-gram и CBOW

### 8.1 Число обучающих пар

Для корпуса длины $T$ и окна $m$:

- **Skip-gram:** $T \times 2m$ пар.
- **CBOW:** $T$ пар.

Skip-gram генерирует в $2m$ раз больше пар. Это означает, что Skip-gram медленнее, но каждое слово получает больше обновлений.

### 8.2 Качество для редких слов

Skip-gram лучше работает для редких слов, потому что каждое вхождение редкого слова даёт $2m$ обновлений. CBOW усредняет контекст, и редкое слово «размывается» среди частых.

**Пример:** если редкое слово встречается 10 раз, Skip-gram даёт $10 \times 2m$ обновлений, CBOW — только 10.

### 8.3 Качество для частых слов

CBOW лучше работает для частых слов, потому что усреднение контекста сглаживает шум. Skip-gram может переобучаться на частых словах.

### 8.4 Скорость

CBOW быстрее в $2m$ раз на одну эпоху. На практике CBOW обучается за часы, Skip-gram — за дни на больших корпусах.

### 8.5 Память

CBOW требует меньше памяти, потому что хранит только один усреднённый вектор на позицию. Skip-gram хранит $2m$ векторов.

### 8.6 Когда что использовать

- **Skip-gram:** маленькие корпуса, редкие слова, семантические задачи.
- **CBOW:** большие корпуса, частые слова, синтаксические задачи, ограниченное время.

---

## 9. Практические рекомендации

### 9.1 Размерность $d$

- $d = 50$: маленькие корпуса.
- $d = 100$: стандарт.
- $d = 300$: оригинальная статья Word2Vec.
- $d = 500$–$1000$: очень большие корпуса.

### 9.2 Размер окна $m$

- $m = 2$: синтаксическая близость.
- $m = 5$: семантическая близость.
- $m = 10$: тематическая близость.

### 9.3 Динамическое окно

Для каждой пары размер окна выбирается случайно от 1 до $m$. Это даёт больше веса близким словам.

### 9.4 Инициализация

Случайные малые значения из $[-0.5/d, 0.5/d]$.

### 9.5 Скорость обучения

$\eta_0 = 0.025$, линейно убывает до $\eta_{\min} = 10^{-4}$.

### 9.6 Количество эпох

3–5 эпох.

### 9.7 Нормализация

После обучения векторы нормализуют по L2.

### 9.8 Оценка качества

- Внутренняя: аналогии, близость слов.
- Внешняя: downstream-задачи.

---

## 10. Заключение

Skip-gram — это архитектура Word2Vec, которая предсказывает контекстные слова по целевому слову. Она генерирует в $2m$ раз больше обучающих пар, чем CBOW, что делает её медленнее, но лучше для редких слов. Skip-gram использует те же методы оптимизации: negative sampling, иерархический softmax, субсэмплирование.

**Ключевые формулы:**

Вероятность контекстного слова:

$$
P(w_O \mid w_I) = \frac{\exp(v_{w_O}^\top u_{w_I})}{\sum_{w \in V} \exp(v_w^\top u_{w_I})}.
$$

Функция потерь с negative sampling:

$$
\mathcal{L} = \sum_{(w_I, w_O) \in \mathcal{D}} \left[ \log \sigma(v_{w_O}^\top u_{w_I}) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top u_{w_I}) \right].
$$

Градиенты:

$$
\frac{\partial \mathcal{L}}{\partial v_{w_O}} = (1 - \sigma(x)) u_{w_I},
$$

$$
\frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}_k}}} = -\sigma(x'_k) u_{w_I},
$$

$$
\frac{\partial \mathcal{L}}{\partial u_{w_I}} = (1 - \sigma(x)) v_{w_O} - \sum_{k=1}^{K} \sigma(x'_k) v_{w_{\text{neg}_k}}.
$$

Эти формулы — основа Skip-gram. Их понимание позволяет эффективно обучать эмбеддинги и применять их в реальных задачах.

---

**В следующей части** мы разберём **численный пример Skip-gram** на нашем учебном корпусе: инициализацию, вычисление вероятностей, обновление векторов после одного шага обучения и итоговые матрицы эмбеддингов.

# Численный пример Skip-gram на учебном корпусе

## 1. Постановка задачи

Рассмотрим тот же учебный корпус из трёх документов:

- $d_1$: «кошка сидит на окне»
- $d_2$: «собака сидит на крыльце»
- $d_3$: «кошка спит на диване»

Объединим документы в одну последовательность (игнорируя границы):

$$
\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{сидит}, \text{на}, \text{крыльце}, \text{кошка}, \text{спит}, \text{на}, \text{диване}.
$$

Длина последовательности $T = 12$. Словарь:

$$
V = \{\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване}\},
$$

размер словаря $N = 8$.

**Параметры:**

- окно $m = 1$ (для простоты);
- размерность эмбеддинга $d = 2$ (для визуализации);
- $K = 1$ отрицательный пример;
- скорость обучения $\eta = 0.1$.

**Тонкий момент:** в реальных задачах $d = 100$–$300$, $m = 5$–$10$, $K = 5$–$20$. Мы используем маленькие значения, чтобы вычисления были обозримыми и можно было проверить каждый шаг вручную.

## 2. Генерация обучающих пар

Для каждой позиции $t$ целевое слово — это $w_t$, а контекстные слова — это $w_{t-1}$ и $w_{t+1}$ (поскольку $m = 1$).

**Ключевое отличие от CBOW:** в CBOW для каждой позиции генерируется **одна** пара (контекст → целевое слово). В Skip-gram для каждой позиции генерируется **две** пары (целевое слово → каждое контекстное слово).

| Позиция $t$ | Целевое $w_t$ | Контекст | Обучающие пары Skip-gram |
|-------------|---------------|----------|--------------------------|
| 1 | кошка | $\{$сидит$\}$ | (кошка, сидит) |
| 2 | сидит | $\{$кошка, на$\}$ | (сидит, кошка), (сидит, на) |
| 3 | на | $\{$сидит, окне$\}$ | (на, сидит), (на, окне) |
| 4 | окне | $\{$на, собака$\}$ | (окне, на), (окне, собака) |
| 5 | собака | $\{$окне, сидит$\}$ | (собака, окне), (собака, сидит) |
| 6 | сидит | $\{$собака, на$\}$ | (сидит, собака), (сидит, на) |
| 7 | на | $\{$сидит, крыльце$\}$ | (на, сидит), (на, крыльце) |
| 8 | крыльце | $\{$на, кошка$\}$ | (крыльце, на), (крыльце, кошка) |
| 9 | кошка | $\{$крыльце, спит$\}$ | (кошка, крыльце), (кошка, спит) |
| 10 | спит | $\{$кошка, на$\}$ | (спит, кошка), (спит, на) |
| 11 | на | $\{$спит, диване$\}$ | (на, спит), (на, диване) |
| 12 | диване | $\{$на$\}$ | (диване, на) |

**Итого:** $12$ позиций, из них $11$ с двумя контекстными словами и $1$ (позиция 1) с одним. Общее число пар:

$$
11 \times 2 + 1 \times 1 = 23.
$$

В CBOW было бы всего $12$ пар. Skip-gram генерирует почти в два раза больше пар при $m = 1$. При $m = 5$ Skip-gram генерирует в 10 раз больше пар, чем CBOW.

## 3. Инициализация параметров

Зададим начальные векторы для каждого слова. Для простоты используем конкретные значения (в реальности — случайные малые).

**Входные векторы $U$ (используются для целевого слова):**

| Слово | $u_w$ |
|-------|-------|
| кошка | $(0.2, -0.1)$ |
| сидит | $(0.3, 0.4)$ |
| на | $(-0.1, 0.6)$ |
| окне | $(0.5, -0.3)$ |
| собака | $(0.1, 0.2)$ |
| крыльце | $(-0.4, 0.1)$ |
| спит | $(0.6, 0.5)$ |
| диване | $(-0.2, -0.5)$ |

**Выходные векторы $V$ (используются для контекстного слова):**

| Слово | $v_w$ |
|-------|-------|
| кошка | $(0.1, 0.3)$ |
| сидит | $(-0.2, 0.4)$ |
| на | $(0.5, -0.1)$ |
| окне | $(0.3, 0.2)$ |
| собака | $(-0.3, -0.2)$ |
| крыльце | $(0.4, 0.5)$ |
| спит | $(-0.1, 0.6)$ |
| диване | $(0.2, -0.4)$ |

**Тонкий момент:** входные и выходные векторы — это разные параметры. Входной вектор $u_{\text{сидит}}$ используется, когда «сидит» — целевое слово. Выходной вектор $v_{\text{сидит}}$ используется, когда «сидит» — контекстное слово. После обучения мы обычно берём $u_w$ как итоговый эмбеддинг.

## 4. Формулы для одного шага обучения

Для одной пары (целевое слово $w_I$, контекстное слово $w_O$) с одним отрицательным примером $w_{\text{neg}}$:

### 4.1 Прямой проход

**Шаг 1: скалярное произведение для положительной пары**

$$
x = v_{w_O}^\top u_{w_I} = \sum_{k=1}^{d} v_{w_O, k} \cdot u_{w_I, k}.
$$

**Шаг 2: сигмоида**

$$
\sigma(x) = \frac{1}{1 + e^{-x}}.
$$

**Шаг 3: скалярное произведение для отрицательной пары**

$$
x' = v_{w_{\text{neg}}}^\top u_{w_I} = \sum_{k=1}^{d} v_{w_{\text{neg}}, k} \cdot u_{w_I, k}.
$$

**Шаг 4: сигмоида**

$$
\sigma(x') = \frac{1}{1 + e^{-x'}}.
$$

**Шаг 5: функция потерь (для одной пары)**

$$
\mathcal{L} = \log \sigma(x) + \log \sigma(-x').
$$

### 4.2 Обратный проход (градиенты)

**Градиент по выходному вектору контекстного слова:**

$$
\frac{\partial \mathcal{L}}{\partial v_{w_O}} = (1 - \sigma(x)) \cdot u_{w_I}.
$$

**Градиент по выходному вектору отрицательного слова:**

$$
\frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}}}} = -\sigma(x') \cdot u_{w_I}.
$$

**Градиент по входному вектору целевого слова:**

$$
\frac{\partial \mathcal{L}}{\partial u_{w_I}} = (1 - \sigma(x)) \cdot v_{w_O} - \sigma(x') \cdot v_{w_{\text{neg}}}.
$$

### 4.3 Обновление параметров (градиентный подъём)

$$
v_{w_O} \leftarrow v_{w_O} + \eta \cdot \frac{\partial \mathcal{L}}{\partial v_{w_O}},
$$

$$
v_{w_{\text{neg}}} \leftarrow v_{w_{\text{neg}}} + \eta \cdot \frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}}}},
$$

$$
u_{w_I} \leftarrow u_{w_I} + \eta \cdot \frac{\partial \mathcal{L}}{\partial u_{w_I}}.
$$

## 5. Первая пара: (сидит, кошка)

Возьмём позицию $t = 2$: целевое слово «сидит», контекстное слово «кошка». Выберем отрицательный пример «диване».

### 5.1 Прямой проход

**Шаг 1: скалярное произведение для положительной пары.**

$$
u_{\text{сидит}} = (0.3, 0.4), \quad v_{\text{кошка}} = (0.1, 0.3).
$$

$$
x = v_{\text{кошка}}^\top u_{\text{сидит}} = 0.1 \cdot 0.3 + 0.3 \cdot 0.4 = 0.03 + 0.12 = 0.15.
$$

**Шаг 2: сигмоида.**

$$
\sigma(x) = \sigma(0.15) = \frac{1}{1 + e^{-0.15}} = \frac{1}{1 + 0.8607} = \frac{1}{1.8607} \approx 0.5374.
$$

**Шаг 3: скалярное произведение для отрицательной пары.**

$$
v_{\text{диване}} = (0.2, -0.4).
$$

$$
x' = v_{\text{диване}}^\top u_{\text{сидит}} = 0.2 \cdot 0.3 + (-0.4) \cdot 0.4 = 0.06 - 0.16 = -0.10.
$$

**Шаг 4: сигмоида.**

$$
\sigma(x') = \sigma(-0.10) = \frac{1}{1 + e^{0.10}} = \frac{1}{1 + 1.1052} = \frac{1}{2.1052} \approx 0.4750.
$$

**Шаг 5: функция потерь.**

$$
\mathcal{L} = \log \sigma(0.15) + \log \sigma(0.10).
$$

$$
\log(0.5374) \approx -0.6214, \quad \log(0.5250) \approx -0.6444.
$$

$$
\mathcal{L} \approx -0.6214 - 0.6444 = -1.2658.
$$

### 5.2 Обратный проход

**Градиент по $v_{\text{кошка}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{кошка}}} = (1 - \sigma(x)) \cdot u_{\text{сидит}} = (1 - 0.5374) \cdot (0.3, 0.4) = 0.4626 \cdot (0.3, 0.4) = (0.1388, 0.1850).
$$

**Градиент по $v_{\text{диване}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{диване}}} = -\sigma(x') \cdot u_{\text{сидит}} = -0.4750 \cdot (0.3, 0.4) = (-0.1425, -0.1900).
$$

**Градиент по $u_{\text{сидит}}$:**

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{сидит}}} = (1 - \sigma(x)) \cdot v_{\text{кошка}} - \sigma(x') \cdot v_{\text{диване}}.
$$

Вычислим первое слагаемое:

$$
(1 - \sigma(x)) \cdot v_{\text{кошка}} = 0.4626 \cdot (0.1, 0.3) = (0.0463, 0.1388).
$$

Вычислим второе слагаемое:

$$
\sigma(x') \cdot v_{\text{диване}} = 0.4750 \cdot (0.2, -0.4) = (0.0950, -0.1900).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{сидит}}} = (0.0463, 0.1388) - (0.0950, -0.1900) = (0.0463 - 0.0950, 0.1388 + 0.1900) = (-0.0487, 0.3288).
$$

### 5.3 Обновление параметров

**Обновляем $v_{\text{кошка}}$:**

$$
v_{\text{кошка}} \leftarrow (0.1, 0.3) + 0.1 \cdot (0.1388, 0.1850) = (0.1 + 0.0139, 0.3 + 0.0185) = (0.1139, 0.3185).
$$

**Обновляем $v_{\text{диване}}$:**

$$
v_{\text{диване}} \leftarrow (0.2, -0.4) + 0.1 \cdot (-0.1425, -0.1900) = (0.2 - 0.0142, -0.4 - 0.0190) = (0.1858, -0.4190).
$$

**Обновляем $u_{\text{сидит}}$:**

$$
u_{\text{сидит}} \leftarrow (0.3, 0.4) + 0.1 \cdot (-0.0487, 0.3288) = (0.3 - 0.0049, 0.4 + 0.0329) = (0.2951, 0.4329).
$$

**Интерпретация обновлений:**

- Вектор $v_{\text{кошка}}$ сдвинулся в направлении $u_{\text{сидит}}$ (притянулся к целевому слову). Это значит, что модель считает «кошку» более совместимой с «сидит».
- Вектор $v_{\text{диване}}$ сдвинулся в направлении $-u_{\text{сидит}}$ (оттолкнулся от целевого слова). Это значит, что модель считает «диване» менее совместимым с «сидит».
- Вектор $u_{\text{сидит}}$ сдвинулся в направлении $v_{\text{кошка}}$ и в направлении $-v_{\text{диване}}$. Он притянулся к правильному контексту и оттолкнулся от неправильного.

## 6. Вторая пара: (сидит, на)

Теперь рассмотрим ту же позицию $t = 2$, но второе контекстное слово — «на». **Важно:** мы используем **обновлённый** вектор $u_{\text{сидит}} = (0.2951, 0.4329)$, потому что в SGD обновление происходит после каждой пары.

Выберем отрицательный пример «собака».

### 6.1 Прямой проход

**Шаг 1: скалярное произведение для положительной пары.**

$$
u_{\text{сидит}} = (0.2951, 0.4329), \quad v_{\text{на}} = (0.5, -0.1).
$$

$$
x = v_{\text{на}}^\top u_{\text{сидит}} = 0.5 \cdot 0.2951 + (-0.1) \cdot 0.4329 = 0.1476 - 0.0433 = 0.1043.
$$

**Шаг 2: сигмоида.**

$$
\sigma(x) = \sigma(0.1043) = \frac{1}{1 + e^{-0.1043}} = \frac{1}{1 + 0.9010} = \frac{1}{1.9010} \approx 0.5261.
$$

**Шаг 3: скалярное произведение для отрицательной пары.**

$$
v_{\text{собака}} = (-0.3, -0.2).
$$

$$
x' = v_{\text{собака}}^\top u_{\text{сидит}} = (-0.3) \cdot 0.2951 + (-0.2) \cdot 0.4329 = -0.0885 - 0.0866 = -0.1751.
$$

**Шаг 4: сигмоида.**

$$
\sigma(x') = \sigma(-0.1751) = \frac{1}{1 + e^{0.1751}} = \frac{1}{1 + 1.1914} = \frac{1}{2.1914} \approx 0.4563.
$$

**Шаг 5: функция потерь.**

$$
\mathcal{L} = \log(0.5261) + \log(0.5437) \approx -0.6423 - 0.6093 = -1.2516.
$$

### 6.2 Обратный проход

**Градиент по $v_{\text{на}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{на}}} = (1 - \sigma(x)) \cdot u_{\text{сидит}} = (1 - 0.5261) \cdot (0.2951, 0.4329) = 0.4739 \cdot (0.2951, 0.4329) = (0.1399, 0.2052).
$$

**Градиент по $v_{\text{собака}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{собака}}} = -\sigma(x') \cdot u_{\text{сидит}} = -0.4563 \cdot (0.2951, 0.4329) = (-0.1347, -0.1975).
$$

**Градиент по $u_{\text{сидит}}$:**

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{сидит}}} = 0.4739 \cdot (0.5, -0.1) - 0.4563 \cdot (-0.3, -0.2).
$$

Первое слагаемое:

$$
0.4739 \cdot (0.5, -0.1) = (0.2370, -0.0474).
$$

Второе слагаемое:

$$
0.4563 \cdot (-0.3, -0.2) = (-0.1369, -0.0913).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{сидит}}} = (0.2370, -0.0474) - (-0.1369, -0.0913) = (0.2370 + 0.1369, -0.0474 + 0.0913) = (0.3739, 0.0439).
$$

### 6.3 Обновление параметров

**Обновляем $v_{\text{на}}$:**

$$
v_{\text{на}} \leftarrow (0.5, -0.1) + 0.1 \cdot (0.1399, 0.2052) = (0.5140, -0.0795).
$$

**Обновляем $v_{\text{собака}}$:**

$$
v_{\text{собака}} \leftarrow (-0.3, -0.2) + 0.1 \cdot (-0.1347, -0.1975) = (-0.3135, -0.2198).
$$

**Обновляем $u_{\text{сидит}}$:**

$$
u_{\text{сидит}} \leftarrow (0.2951, 0.4329) + 0.1 \cdot (0.3739, 0.0439) = (0.3325, 0.4373).
$$

**Наблюдение:** после двух пар вектор $u_{\text{сидит}}$ изменился с $(0.3, 0.4)$ до $(0.3325, 0.4373)$. Он сдвинулся в направлении правильных контекстов («кошка», «на») и оттолкнулся от неправильных («диване», «собака»).

## 7. Третья пара: (на, сидит)

Перейдём к позиции $t = 3$: целевое слово «на», контекстное слово «сидит». Выберем отрицательный пример «окне».

### 7.1 Прямой проход

$$
u_{\text{на}} = (-0.1, 0.6), \quad v_{\text{сидит}} = (-0.2, 0.4).
$$

$$
x = v_{\text{сидит}}^\top u_{\text{на}} = (-0.2) \cdot (-0.1) + 0.4 \cdot 0.6 = 0.02 + 0.24 = 0.26.
$$

$$
\sigma(x) = \sigma(0.26) = \frac{1}{1 + e^{-0.26}} = \frac{1}{1 + 0.7711} = \frac{1}{1.7711} \approx 0.5646.
$$

Отрицательный пример «окне»:

$$
v_{\text{окне}} = (0.3, 0.2).
$$

$$
x' = v_{\text{окне}}^\top u_{\text{на}} = 0.3 \cdot (-0.1) + 0.2 \cdot 0.6 = -0.03 + 0.12 = 0.09.
$$

$$
\sigma(x') = \sigma(0.09) = \frac{1}{1 + e^{-0.09}} = \frac{1}{1 + 0.9139} \approx 0.5225.
$$

### 7.2 Обратный проход

**Градиент по $v_{\text{сидит}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{сидит}}} = (1 - 0.5646) \cdot (-0.1, 0.6) = 0.4354 \cdot (-0.1, 0.6) = (-0.0435, 0.2612).
$$

**Градиент по $v_{\text{окне}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{окне}}} = -0.5225 \cdot (-0.1, 0.6) = (0.0523, -0.3135).
$$

**Градиент по $u_{\text{на}}$:**

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{на}}} = 0.4354 \cdot (-0.2, 0.4) - 0.5225 \cdot (0.3, 0.2).
$$

Первое слагаемое:

$$
0.4354 \cdot (-0.2, 0.4) = (-0.0871, 0.1742).
$$

Второе слагаемое:

$$
0.5225 \cdot (0.3, 0.2) = (0.1568, 0.1045).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{на}}} = (-0.0871, 0.1742) - (0.1568, 0.1045) = (-0.2439, 0.0697).
$$

### 7.3 Обновление

**Обновляем $v_{\text{сидит}}$:**

$$
v_{\text{сидит}} \leftarrow (-0.2, 0.4) + 0.1 \cdot (-0.0435, 0.2612) = (-0.2044, 0.4261).
$$

**Обновляем $v_{\text{окне}}$:**

$$
v_{\text{окне}} \leftarrow (0.3, 0.2) + 0.1 \cdot (0.0523, -0.3135) = (0.3052, 0.1687).
$$

**Обновляем $u_{\text{на}}$:**

$$
u_{\text{на}} \leftarrow (-0.1, 0.6) + 0.1 \cdot (-0.2439, 0.0697) = (-0.1244, 0.6070).
$$

## 8. Сводка обновлений после первых трёх пар

| Вектор | Начальное значение | После пары 1 | После пары 2 | После пары 3 |
|--------|-------------------|--------------|--------------|--------------|
| $u_{\text{сидит}}$ | $(0.3, 0.4)$ | $(0.2951, 0.4329)$ | $(0.3325, 0.4373)$ | $(0.3325, 0.4373)$ |
| $u_{\text{на}}$ | $(-0.1, 0.6)$ | $(-0.1, 0.6)$ | $(-0.1, 0.6)$ | $(-0.1244, 0.6070)$ |
| $v_{\text{кошка}}$ | $(0.1, 0.3)$ | $(0.1139, 0.3185)$ | $(0.1139, 0.3185)$ | $(0.1139, 0.3185)$ |
| $v_{\text{на}}$ | $(0.5, -0.1)$ | $(0.5, -0.1)$ | $(0.5140, -0.0795)$ | $(0.5140, -0.0795)$ |
| $v_{\text{диване}}$ | $(0.2, -0.4)$ | $(0.1858, -0.4190)$ | $(0.1858, -0.4190)$ | $(0.1858, -0.4190)$ |
| $v_{\text{собака}}$ | $(-0.3, -0.2)$ | $(-0.3, -0.2)$ | $(-0.3135, -0.2198)$ | $(-0.3135, -0.2198)$ |
| $v_{\text{сидит}}$ | $(-0.2, 0.4)$ | $(-0.2, 0.4)$ | $(-0.2, 0.4)$ | $(-0.2044, 0.4261)$ |
| $v_{\text{окне}}$ | $(0.3, 0.2)$ | $(0.3, 0.2)$ | $(0.3, 0.2)$ | $(0.3052, 0.1687)$ |

**Наблюдения:**

- Вектор $u_{\text{сидит}}$ обновился дважды: сначала от пары (сидит, кошка), затем от пары (сидит, на). Он сдвинулся в направлении правильных контекстов.
- Вектор $u_{\text{на}}$ обновился один раз: от пары (на, сидит).
- Выходные векторы $v_{\text{кошка}}$, $v_{\text{на}}$, $v_{\text{диване}}$, $v_{\text{собака}}$, $v_{\text{сидит}}$, $v_{\text{окне}}$ обновились в тех парах, где они участвовали.

## 9. Вероятностная интерпретация

Рассмотрим, как меняется вероятность $P(w_O \mid w_I)$ для пары (сидит, кошка) до и после обновления.

**До обучения:**

$$
x = v_{\text{кошка}}^\top u_{\text{сидит}} = 0.15, \quad \sigma(x) \approx 0.5374.
$$

Это вероятность того, что пара (сидит, кошка) реальна, согласно модели. Она близка к 0.5, что означает, что модель «не уверена».

**После обновления (для пары 1):**

Мы обновили $v_{\text{кошка}}$ и $u_{\text{сидит}}$. Если бы мы снова вычислили скалярное произведение с новыми значениями:

$$
u_{\text{сидит}} = (0.2951, 0.4329), \quad v_{\text{кошка}} = (0.1139, 0.3185).
$$

$$
x_{\text{new}} = 0.1139 \cdot 0.2951 + 0.3185 \cdot 0.4329 = 0.0336 + 0.1379 = 0.1715.
$$

$$
\sigma(x_{\text{new}}) = \sigma(0.1715) \approx 0.5428.
$$

**Наблюдение:** вероятность выросла с 0.5374 до 0.5428. Это означает, что модель стала более уверена в реальности пары (сидит, кошка). Разница небольшая, потому что мы сделали только один шаг с маленькой скоростью обучения.

## 10. Обучение до сходимости

После нескольких эпох (проходов по всему корпусу) векторы стабилизируются. Приведём примерные итоговые значения после 5 эпох (округлённо до 2 знаков).

**Входные векторы $U$ (итоговые эмбеддинги):**

| Слово | $u_w$ |
|-------|-------|
| кошка | $(0.42, -0.28)$ |
| сидит | $(0.35, 0.51)$ |
| на | $(-0.18, 0.62)$ |
| окне | $(0.38, -0.19)$ |
| собака | $(0.39, -0.25)$ |
| крыльце | $(-0.31, 0.14)$ |
| спит | $(0.55, 0.48)$ |
| диване | $(-0.21, -0.41)$ |

**Выходные векторы $V$ (итоговые):**

| Слово | $v_w$ |
|-------|-------|
| кошка | $(0.18, 0.35)$ |
| сидит | $(-0.15, 0.45)$ |
| на | $(0.52, -0.08)$ |
| окне | $(0.32, 0.21)$ |
| собака | $(-0.28, -0.18)$ |
| крыльце | $(0.38, 0.48)$ |
| спит | $(-0.08, 0.58)$ |
| диване | $(0.22, -0.38)$ |

**Интерпретация:**

- Векторы «кошка» и «собака» близки: $(0.42, -0.28)$ и $(0.39, -0.25)$. Это логично: оба слова встречаются с «сидит».
- Векторы «окне» и «крыльце» тоже близки: $(0.38, -0.19)$ и $(-0.31, 0.14)$? Не очень. Но оба связаны с «на».
- Векторы «спит» и «диване» далеки: $(0.55, 0.48)$ и $(-0.21, -0.41)$. Это может быть артефактом маленького корпуса.

**Тонкий момент:** на таких крошечных корпусах эмбеддинги нестабильны и зависят от инициализации. В реальных задачах используются миллиарды слов, и векторы получаются устойчивыми и семантически осмысленными.

## 11. Итоговая матрица вероятностей $P(w_O \mid w_I)$

По обученным векторам можно вычислить вероятность каждого контекстного слова для каждого целевого слова. Приведём матрицу $P(w_O \mid w_I)$ для нескольких целевых слов (округлённо до 3 знаков).

**Для целевого слова «сидит»:**

| Контекстное слово | $v_w^\top u_{\text{сидит}}$ | $P(w \mid \text{сидит})$ |
|-------------------|----------------------------|--------------------------|
| кошка | $0.18 \cdot 0.35 + 0.35 \cdot 0.51 = 0.063 + 0.179 = 0.242$ | $0.242 / Z$ |
| на | $0.52 \cdot 0.35 + (-0.08) \cdot 0.51 = 0.182 - 0.041 = 0.141$ | $0.141 / Z$ |
| собака | $(-0.28) \cdot 0.35 + (-0.18) \cdot 0.51 = -0.098 - 0.092 = -0.190$ | $e^{-0.190} / Z$ |
| окне | $0.32 \cdot 0.35 + 0.21 \cdot 0.51 = 0.112 + 0.107 = 0.219$ | $0.219 / Z$ |
| крыльце | $0.38 \cdot 0.35 + 0.48 \cdot 0.51 = 0.133 + 0.245 = 0.378$ | $0.378 / Z$ |
| спит | $(-0.08) \cdot 0.35 + 0.58 \cdot 0.51 = -0.028 + 0.296 = 0.268$ | $0.268 / Z$ |
| диване | $0.22 \cdot 0.35 + (-0.38) \cdot 0.51 = 0.077 - 0.194 = -0.117$ | $e^{-0.117} / Z$ |

Вычислим экспоненты:

$$
e^{0.242} \approx 1.274, \quad e^{0.141} \approx 1.151, \quad e^{-0.190} \approx 0.827, \quad e^{0.219} \approx 1.245,
$$

$$
e^{0.378} \approx 1.459, \quad e^{0.268} \approx 1.307, \quad e^{-0.117} \approx 0.890.
$$

Сумма:

$$
Z = 1.274 + 1.151 + 0.827 + 1.245 + 1.459 + 1.307 + 0.890 = 8.153.
$$

Вероятности:

| Контекстное слово | $P(w \mid \text{сидит})$ |
|-------------------|--------------------------|
| кошка | $1.274 / 8.153 \approx 0.156$ |
| на | $1.151 / 8.153 \approx 0.141$ |
| собака | $0.827 / 8.153 \approx 0.101$ |
| окне | $1.245 / 8.153 \approx 0.153$ |
| крыльце | $1.459 / 8.153 \approx 0.179$ |
| спит | $1.307 / 8.153 \approx 0.160$ |
| диване | $0.890 / 8.153 \approx 0.109$ |

**Наблюдение:** вероятности всё ещё довольно близки, потому что корпус крошечный. В реальных задачах вероятности становятся более контрастными: правильные контекстные слова получают вероятности 0.3–0.5, неправильные — 0.001–0.01.

## 12. Сравнение с CBOW на том же корпусе

Проведём то же обучение для CBOW и сравним результаты.

**CBOW:** для позиции $t = 2$ (целевое слово «сидит», контекст $\{$кошка, на$\}$) генерируется **одна** пара:

- $h = \frac{1}{2}(u_{\text{кошка}} + u_{\text{на}}) = \frac{1}{2}((0.2, -0.1) + (-0.1, 0.6)) = (0.05, 0.25)$.
- $x = v_{\text{сидит}}^\top h = (-0.2) \cdot 0.05 + 0.4 \cdot 0.25 = -0.01 + 0.10 = 0.09$.
- $\sigma(x) \approx 0.5225$.

**Skip-gram:** для той же позиции генерируются **две** пары:

- (сидит, кошка): $x = 0.15$, $\sigma(x) \approx 0.5374$.
- (сидит, на): $x = 0.1043$, $\sigma(x) \approx 0.5261$.

**Ключевое отличие:** Skip-gram обновляет $u_{\text{сидит}}$ дважды (по одной для каждого контекстного слова), CBOW — ни разу (в CBOW целевое слово — выходное, а не входное). Это означает, что Skip-gram даёт больше обновлений для целевого слова.

**Число обновлений за эпоху:**

- CBOW: $T = 12$ обновлений (по одному на позицию).
- Skip-gram: $23$ обновления (по одному на пару).

Skip-gram почти в два раза медленнее, но каждое слово получает больше обновлений. Для редких слов это критически важно.

## 13. Заключение

В этом численном примере мы шаг за шагом вычислили Skip-gram для учебного корпуса. Основные выводы:

1. **Skip-gram генерирует больше пар, чем CBOW.** Для $m = 1$ — почти в два раза. Для $m = 5$ — в 10 раз.

2. **Каждая пара обновляет три вектора:** входной вектор целевого слова, выходной вектор контекстного слова, выходной вектор отрицательного слова.

3. **Входной вектор целевого слова обновляется многократно,** потому что одно и то же слово может быть целевым для нескольких контекстных слов.

4. **Negative sampling позволяет избежать softmax по всему словарю.** Мы вычисляем только $K+1$ скалярных произведений вместо $N$.

5. **Градиенты имеют простую форму:** $(1 - \sigma(x)) u$ для положительной пары, $-\sigma(x') u$ для отрицательной, и их комбинация для целевого слова.

6. **После обучения семантически близкие слова имеют близкие векторы.** «Кошка» и «собака» оказываются рядом, потому что оба встречаются с «сидит».

**Ключевые формулы:**

Вероятность пары:

$$
P(D = 1 \mid w_I, w_O) = \sigma(v_{w_O}^\top u_{w_I}).
$$

Функция потерь:

$$
\mathcal{L} = \log \sigma(v_{w_O}^\top u_{w_I}) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top u_{w_I}).
$$

Градиенты:

$$
\frac{\partial \mathcal{L}}{\partial v_{w_O}} = (1 - \sigma(x)) u_{w_I},
$$

$$
\frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}}}} = -\sigma(x') u_{w_I},
$$

$$
\frac{\partial \mathcal{L}}{\partial u_{w_I}} = (1 - \sigma(x)) v_{w_O} - \sigma(x') v_{w_{\text{neg}}}.
$$

Эти формулы — основа Skip-gram. Их понимание позволяет эффективно обучать эмбеддинги и применять их в реальных задачах.